# 600 — Program–Drug Associations

## Objective

Characterize lineage-aware associations between the frozen Phase 4 consensus programs and pharmacogenomic drug-response measurements across GDSC, CTRP, and PRISM.

This notebook treats pharmacogenomic results as computational associations describing resistance-like contexts. It does not establish clinical resistance, therapeutic efficacy, causal mechanisms, or validated biomarkers.

## Analytical boundary

Before any program–drug association is estimated, this notebook will:

1. validate the frozen upstream program and cell-model handoffs;
2. characterize identifier compatibility and model coverage across GDSC, CTRP, and PRISM;
3. characterize drug and screen coverage without using association results to select drugs, models, metrics, thresholds, or lineages;
4. determine whether external program scores can be obtained by direct overlap or by deterministic projection of frozen Phase 4 gene weights, without refitting, reorientation, or reweighting;
5. document the information required to freeze the prospective Phase 6 analysis contract.

No inferential program–drug results will be inspected before the applicable Phase 6 analytical rules are frozen.

## Evidence roles

- **GDSC:** developmental/internal pharmacogenomic characterization. GDSC contributed to upstream Phase 3 analyses and is therefore not treated as independent external replication.
- **CTRP:** external cross-screen pharmacogenomic resource.
- **PRISM:** external cross-screen pharmacogenomic resource, preserving screen and compound-batch structure until prospective aggregation rules are defined.

Cell-line overlap, lineage structure, drug identity, drug-family structure, screen structure, platform differences, and proliferation-related confounding will remain explicit throughout Phase 6.

Phase 5 functional-vulnerability results will not be used to select drugs, programs, models, thresholds, or hypotheses for this analysis.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import zipfile
import hashlib
import json

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.paths import Paths
from pancancer_epigenetics.utils.raw_data_registry import load_raw_data_registry

In [2]:
# =============================================================================
# Load project registries
# =============================================================================

artifact_registry = load_artifact_registry()
raw_registry = load_raw_data_registry()

print(f"Frozen artifacts registered: {len(artifact_registry['artifacts'])}")
print(
    "Pharmacogenomic raw resources registered:",
    ", ".join(resource for resource in ("gdsc", "ctrp", "prism") if resource in raw_registry),
)

Frozen artifacts registered: 109
Pharmacogenomic raw resources registered: gdsc, ctrp, prism


In [3]:
# =============================================================================
# Resolve frozen upstream artifact paths
# =============================================================================

UPSTREAM_ARTIFACT_IDS = (
    "phase3.302.integrated_modeling_cohort",
    "phase3.303.harmonized_expression",
    "phase4.400.cross_system_shared_gene_universe",
    "phase4.401.consensus_transcriptomic_program_catalog",
    "phase4.401.consensus_cellline_scores",
    "phase4.401.consensus_transcriptomic_gene_weights",
)

upstream_artifact_paths = {
    artifact_id: resolve_artifact_path(artifact_registry, artifact_id)
    for artifact_id in UPSTREAM_ARTIFACT_IDS
}

for artifact_id in UPSTREAM_ARTIFACT_IDS:
    print(
        f"{artifact_id}: "
        f"{artifact_registry['artifacts'][artifact_id]['path']}"
    )

phase3.302.integrated_modeling_cohort: data/interim/metadata/302_integrated_modeling_cohort.csv
phase3.303.harmonized_expression: data/interim/expression/303_expression_harmonized.parquet
phase4.400.cross_system_shared_gene_universe: data/processed/consensus_programs/400_cross_system_shared_gene_universe.csv
phase4.401.consensus_transcriptomic_program_catalog: data/processed/consensus_programs/401_consensus_transcriptomic_program_catalog.csv
phase4.401.consensus_cellline_scores: data/processed/consensus_programs/401_consensus_cellline_scores.parquet
phase4.401.consensus_transcriptomic_gene_weights: data/processed/consensus_programs/401_consensus_transcriptomic_gene_weights.csv


In [4]:
# =============================================================================
# Validate frozen upstream handoffs
# =============================================================================

upstream_handoff_status = pd.DataFrame(
    [
        {
            "artifact_id": artifact_id,
            "status": artifact_registry["artifacts"][artifact_id]["status"],
            "exists": upstream_artifact_paths[artifact_id].is_file(),
        }
        for artifact_id in UPSTREAM_ARTIFACT_IDS
    ]
)

print(
    "All inputs frozen:",
    upstream_handoff_status["status"].eq("frozen").all(),
)
print(
    "All input files present:",
    upstream_handoff_status["exists"].all(),
)

upstream_handoff_status

All inputs frozen: True
All input files present: True


,artifact_id,status,exists
0,phase3.302.integrated_modeling_cohort,frozen,True
1,phase3.303.harmonized_expression,frozen,True
2,phase4.400.cross_system_shared_gene_universe,frozen,True
3,phase4.401.consensus_transcriptomic_program_ca...,frozen,True
4,phase4.401.consensus_cellline_scores,frozen,True
5,phase4.401.consensus_transcriptomic_gene_weights,frozen,True


In [5]:
# =============================================================================
# Load frozen cell-model cohort and consensus-program scores
# =============================================================================

modeling_cohort = pd.read_csv(
    upstream_artifact_paths["phase3.302.integrated_modeling_cohort"]
)
consensus_cellline_scores = pd.read_parquet(
    upstream_artifact_paths["phase4.401.consensus_cellline_scores"]
)

print("Modeling cohort shape:", modeling_cohort.shape)
print("Consensus cell-line scores shape:", consensus_cellline_scores.shape)

print("\nModeling cohort columns:")
print(modeling_cohort.columns.tolist())

print("\nConsensus score columns:")
print(consensus_cellline_scores.columns.tolist())

Modeling cohort shape: (713, 8)
Consensus cell-line scores shape: (713, 4)

Modeling cohort columns:
['ModelID', 'SangerModelID', 'COSMICID', 'CellLineName', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype', 'CCLEName']

Consensus score columns:
['ModelID', 'CONSENSUS_TX_01', 'CONSENSUS_TX_02', 'CONSENSUS_TX_03']


In [6]:
# =============================================================================
# Validate cell-model identity across frozen handoffs
# =============================================================================

modeling_ids = set(modeling_cohort["ModelID"])
score_ids = set(consensus_cellline_scores["ModelID"])

print("Modeling cohort unique ModelID:", modeling_cohort["ModelID"].nunique())
print(
    "Consensus scores unique ModelID:",
    consensus_cellline_scores["ModelID"].nunique(),
)
print(
    "Duplicate ModelID in modeling cohort:",
    modeling_cohort["ModelID"].duplicated().sum(),
)
print(
    "Duplicate ModelID in consensus scores:",
    consensus_cellline_scores["ModelID"].duplicated().sum(),
)
print("ModelID sets identical:", modeling_ids == score_ids)
print("Models only in cohort:", len(modeling_ids - score_ids))
print("Models only in scores:", len(score_ids - modeling_ids))

Modeling cohort unique ModelID: 713
Consensus scores unique ModelID: 713
Duplicate ModelID in modeling cohort: 0
Duplicate ModelID in consensus scores: 0
ModelID sets identical: True
Models only in cohort: 0
Models only in scores: 0


In [7]:
# =============================================================================
# Assemble frozen Phase 6 anchor cohort
# =============================================================================

phase6_anchor_cohort = modeling_cohort.merge(
    consensus_cellline_scores,
    on="ModelID",
    how="inner",
    validate="one_to_one",
)

print("Phase 6 anchor cohort shape:", phase6_anchor_cohort.shape)
print(
    "Models retained:",
    phase6_anchor_cohort["ModelID"].nunique(),
)

phase6_anchor_cohort.head()

Phase 6 anchor cohort shape: (713, 11)
Models retained: 713


,ModelID,SangerModelID,COSMICID,CellLineName,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,CCLEName,CONSENSUS_TX_01,CONSENSUS_TX_02,CONSENSUS_TX_03
0,ACH-000001,SIDM00105,905933.0,NIH:OVCAR-3,Ovary/Fallopian Tube,Ovarian Epithelial Tumor,High-Grade Serous Ovarian Cancer,NIHOVCAR3_OVARY,-0.618908,0.121723,-0.047979
1,ACH-000002,SIDM00829,905938.0,HL-60,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,1.312391,0.062750,-0.407409
2,ACH-000004,SIDM00594,907053.0,HEL,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,HEL_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,1.236019,0.613855,-0.498685
3,ACH-000006,SIDM01023,908148.0,MONO-MAC-6,Myeloid,Acute Myeloid Leukemia,Acute Monoblastic/Monocytic Leukemia,MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,1.496321,0.404348,-0.515095
4,ACH-000007,SIDM00677,907795.0,LS513,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,LS513_LARGE_INTESTINE,-0.238521,-1.263317,-1.175949


In [8]:
# =============================================================================
# Characterize anchor-cohort lineage structure
# =============================================================================

lineage_counts = (
    phase6_anchor_cohort["OncotreeLineage"]
    .value_counts(dropna=False)
    .rename_axis("OncotreeLineage")
    .reset_index(name="n_models")
)

print("Number of lineages:", lineage_counts.shape[0])
print(
    "Models with missing lineage:",
    phase6_anchor_cohort["OncotreeLineage"].isna().sum(),
)
print("Smallest lineage size:", lineage_counts["n_models"].min())
print("Largest lineage size:", lineage_counts["n_models"].max())

lineage_counts

Number of lineages: 27
Models with missing lineage: 0
Smallest lineage size: 1
Largest lineage size: 136


,OncotreeLineage,n_models
0,Lung,136
1,Lymphoid,86
2,Esophagus/Stomach,51
3,Breast,47
4,Bowel,43
5,Skin,37
6,CNS/Brain,37
7,Ovary/Fallopian Tube,34
8,Myeloid,32
9,Pancreas,28


In [9]:
# =============================================================================
# Resolve Phase 6 pharmacogenomic input paths
# =============================================================================

PHARMACOGENOMIC_INPUT_ROLES = {
    "gdsc": {
        "response": "pharmacogenomic_response_matrix",
    },
    "ctrp": {
        "response_archive": "primary_pharmacogenomic_archive",
    },
    "prism": {
        "cell_line_metadata": "cell_line_metadata",
        "response": "dose_response_curve_parameters",
        "treatment_metadata": "replicate_collapsed_treatment_metadata",
    },
}

pharmacogenomic_input_paths = {}

for resource_id, inputs in PHARMACOGENOMIC_INPUT_ROLES.items():
    resource = raw_registry[resource_id]

    for input_name, role in inputs.items():
        matches = [
            file_name
            for file_name, metadata in resource["files"].items()
            if metadata.get("role") == role
        ]

        if len(matches) != 1:
            raise ValueError(
                f"{resource_id}: expected one file for role '{role}', "
                f"found {len(matches)}"
            )

        pharmacogenomic_input_paths[(resource_id, input_name)] = (
            Paths.root / resource["canonical_dir"] / matches[0]
        )

for key, path in pharmacogenomic_input_paths.items():
    print(f"{key}: {path.relative_to(Paths.root)}")

('gdsc', 'response'): data\raw\gdsc\GDSC2_fitted_dose_response_27Oct23.xlsx
('ctrp', 'response_archive'): data\raw\ctrp\CTRPv2.0_2015_ctd2_ExpandedDataset.zip
('prism', 'cell_line_metadata'): data\raw\prism\secondary-screen-cell-line-info.csv
('prism', 'response'): data\raw\prism\secondary-screen-dose-response-curve-parameters.csv
('prism', 'treatment_metadata'): data\raw\prism\secondary-screen-replicate-collapsed-treatment-info.csv


In [10]:
# =============================================================================
# Quantify direct-ID model coverage in GDSC and PRISM
# =============================================================================

gdsc_models = (
    pd.read_excel(
        pharmacogenomic_input_paths[("gdsc", "response")],
        usecols=["SANGER_MODEL_ID"],
    )
    .dropna()
    .drop_duplicates()
)

prism_models = (
    pd.read_csv(
        pharmacogenomic_input_paths[("prism", "response")],
        usecols=["depmap_id", "screen_id"],
    )
    .dropna(subset=["depmap_id"])
    .drop_duplicates()
)

anchor_sanger_ids = set(phase6_anchor_cohort["SangerModelID"].dropna())
anchor_model_ids = set(phase6_anchor_cohort["ModelID"])

gdsc_response_ids = set(gdsc_models["SANGER_MODEL_ID"])
gdsc_anchor_overlap = gdsc_response_ids & anchor_sanger_ids

prism_coverage_rows = []

for screen_id, screen_models in prism_models.groupby("screen_id"):
    response_ids = set(screen_models["depmap_id"])
    overlap_ids = response_ids & anchor_model_ids

    prism_coverage_rows.append(
        {
            "resource": f"PRISM {screen_id}",
            "response_models": len(response_ids),
            "anchor_overlap": len(overlap_ids),
            "anchor_coverage_pct": 100 * len(overlap_ids) / len(anchor_model_ids),
        }
    )

direct_id_coverage = pd.DataFrame(
    [
        {
            "resource": "GDSC",
            "response_models": len(gdsc_response_ids),
            "anchor_overlap": len(gdsc_anchor_overlap),
            "anchor_coverage_pct": 100 * len(gdsc_anchor_overlap) / len(anchor_sanger_ids),
        },
        *prism_coverage_rows,
    ]
)

direct_id_coverage

,resource,response_models,anchor_overlap,anchor_coverage_pct
0,GDSC,969,713,100.000000
1,PRISM HTS002,480,341,47.826087
2,PRISM MTS005,444,315,44.179523
3,PRISM MTS006,476,338,47.405330
4,PRISM MTS010,473,335,46.984572


In [11]:
# =============================================================================
# Assess deterministic CTRP-to-anchor name compatibility
# =============================================================================

with zipfile.ZipFile(
    pharmacogenomic_input_paths[("ctrp", "response_archive")]
) as ctrp_archive:
    ctrp_cell_lines = pd.read_csv(
        ctrp_archive.open("v20.meta.per_cell_line.txt"),
        sep="\t",
        usecols=["master_ccl_id", "ccl_name"],
    )

ctrp_cell_lines["name_key"] = (
    ctrp_cell_lines["ccl_name"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

anchor_name_map = (
    phase6_anchor_cohort[
        ["ModelID", "CellLineName"]
    ]
    .copy()
)

anchor_name_map["name_key"] = (
    anchor_name_map["CellLineName"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

ctrp_key_counts = ctrp_cell_lines["name_key"].value_counts()
anchor_key_counts = anchor_name_map["name_key"].value_counts()

shared_keys = set(ctrp_key_counts.index) & set(anchor_key_counts.index)

unambiguous_shared_keys = {
    key
    for key in shared_keys
    if ctrp_key_counts[key] == 1
    and anchor_key_counts[key] == 1
}

print("CTRP cell lines:", ctrp_cell_lines["master_ccl_id"].nunique())
print("Anchor models:", anchor_name_map["ModelID"].nunique())
print("Shared normalized name keys:", len(shared_keys))
print(
    "Unambiguous one-to-one shared keys:",
    len(unambiguous_shared_keys),
)
print(
    "Shared keys with CTRP collisions:",
    sum(ctrp_key_counts[key] > 1 for key in shared_keys),
)
print(
    "Shared keys with anchor collisions:",
    sum(anchor_key_counts[key] > 1 for key in shared_keys),
)

CTRP cell lines: 1107
Anchor models: 713
Shared normalized name keys: 623
Unambiguous one-to-one shared keys: 623
Shared keys with CTRP collisions: 0
Shared keys with anchor collisions: 0


In [12]:
# =============================================================================
# Quantify CTRP response-covered anchor models
# =============================================================================

ctrp_anchor_crosswalk = (
    ctrp_cell_lines.loc[
        ctrp_cell_lines["name_key"].isin(unambiguous_shared_keys),
        ["master_ccl_id", "ccl_name", "name_key"],
    ]
    .merge(
        anchor_name_map.loc[
            anchor_name_map["name_key"].isin(unambiguous_shared_keys),
            ["ModelID", "CellLineName", "name_key"],
        ],
        on="name_key",
        how="inner",
        validate="one_to_one",
    )
)

with zipfile.ZipFile(
    pharmacogenomic_input_paths[("ctrp", "response_archive")]
) as ctrp_archive:
    ctrp_response_experiments = pd.read_csv(
        ctrp_archive.open("v20.data.curves_post_qc.txt"),
        sep="\t",
        usecols=["experiment_id"],
    )
    ctrp_experiment_map = (
        pd.read_csv(
            ctrp_archive.open("v20.meta.per_experiment.txt"),
            sep="\t",
            usecols=["experiment_id", "master_ccl_id"],
        )
        .drop_duplicates()
    )

response_master_ccl_ids = set(
    ctrp_response_experiments
    .merge(
        ctrp_experiment_map,
        on="experiment_id",
        how="left",
        validate="many_to_one",
    )["master_ccl_id"]
    .dropna()
)

ctrp_response_crosswalk = ctrp_anchor_crosswalk.loc[
    ctrp_anchor_crosswalk["master_ccl_id"].isin(
        response_master_ccl_ids
    )
].copy()

print("Deterministic CTRP-anchor matches:", len(ctrp_anchor_crosswalk))
print(
    "Matches represented in post-QC response:",
    len(ctrp_response_crosswalk),
)
print(
    "Anchor models without deterministic CTRP response coverage:",
    len(anchor_model_ids - set(ctrp_response_crosswalk["ModelID"])),
)

Deterministic CTRP-anchor matches: 623
Matches represented in post-QC response: 566
Anchor models without deterministic CTRP response coverage: 147


In [13]:
# =============================================================================
# Characterize CTRP anchor-coverage loss
# =============================================================================

ctrp_matched_model_ids = set(ctrp_anchor_crosswalk["ModelID"])
ctrp_response_model_ids = set(ctrp_response_crosswalk["ModelID"])

ctrp_coverage_status = phase6_anchor_cohort[
    ["ModelID", "CellLineName", "OncotreeLineage"]
].copy()

ctrp_coverage_status["ctrp_name_match"] = (
    ctrp_coverage_status["ModelID"].isin(ctrp_matched_model_ids)
)
ctrp_coverage_status["ctrp_post_qc_response"] = (
    ctrp_coverage_status["ModelID"].isin(ctrp_response_model_ids)
)

ctrp_coverage_status["coverage_status"] = "no_deterministic_name_match"
ctrp_coverage_status.loc[
    ctrp_coverage_status["ctrp_name_match"],
    "coverage_status",
] = "matched_without_post_qc_response"
ctrp_coverage_status.loc[
    ctrp_coverage_status["ctrp_post_qc_response"],
    "coverage_status",
] = "response_covered"

print(
    ctrp_coverage_status["coverage_status"]
    .value_counts()
)

print(
    "\nCTRP anchor response coverage:",
    f"{100 * len(ctrp_response_model_ids) / len(anchor_model_ids):.2f}%",
)

coverage_status
response_covered                    566
no_deterministic_name_match          90
matched_without_post_qc_response     57
Name: count, dtype: int64

CTRP anchor response coverage: 79.38%


In [14]:
# =============================================================================
# Evaluate secondary deterministic CTRP model mapping
# =============================================================================

unmatched_anchor_ids = (
    anchor_model_ids - ctrp_matched_model_ids
)

depmap_stripped_names = pd.read_csv(
    Paths.depmap / "Model.csv",
    usecols=["ModelID", "StrippedCellLineName"],
)

depmap_stripped_names = (
    depmap_stripped_names.loc[
        depmap_stripped_names["ModelID"].isin(unmatched_anchor_ids)
    ]
    .dropna(subset=["StrippedCellLineName"])
    .copy()
)

depmap_stripped_names["stripped_name_key"] = (
    depmap_stripped_names["StrippedCellLineName"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

used_ctrp_ids = set(
    ctrp_anchor_crosswalk["master_ccl_id"]
)

available_ctrp_cell_lines = (
    ctrp_cell_lines.loc[
        ~ctrp_cell_lines["master_ccl_id"].isin(used_ctrp_ids)
    ]
    .copy()
)

available_ctrp_cell_lines["stripped_name_key"] = (
    available_ctrp_cell_lines["ccl_name"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

anchor_key_counts = (
    depmap_stripped_names["stripped_name_key"]
    .value_counts()
)
ctrp_key_counts = (
    available_ctrp_cell_lines["stripped_name_key"]
    .value_counts()
)

shared_stripped_keys = (
    set(anchor_key_counts.index)
    & set(ctrp_key_counts.index)
)

unambiguous_stripped_keys = {
    key
    for key in shared_stripped_keys
    if anchor_key_counts[key] == 1
    and ctrp_key_counts[key] == 1
}

print("Previously unmatched anchor models:", len(unmatched_anchor_ids))
print(
    "Models with available StrippedCellLineName:",
    depmap_stripped_names["ModelID"].nunique(),
)
print(
    "Shared stripped-name keys:",
    len(shared_stripped_keys),
)
print(
    "Unambiguous secondary matches:",
    len(unambiguous_stripped_keys),
)
print(
    "Ambiguous shared keys:",
    len(shared_stripped_keys - unambiguous_stripped_keys),
)

Previously unmatched anchor models: 90
Models with available StrippedCellLineName: 90
Shared stripped-name keys: 0
Unambiguous secondary matches: 0
Ambiguous shared keys: 0


In [15]:
# =============================================================================
# Construct anchor-restricted CTRP model crosswalk
# =============================================================================

ctrp_model_crosswalk = (
    ctrp_response_crosswalk[
        [
            "master_ccl_id",
            "ccl_name",
            "ModelID",
            "CellLineName",
        ]
    ]
    .assign(mapping_method="exact_normalized_cell_line_name")
    .sort_values("ModelID")
    .reset_index(drop=True)
)

print("Final CTRP mapped models:", len(ctrp_model_crosswalk))
print(
    "Unique ModelID:",
    ctrp_model_crosswalk["ModelID"].nunique(),
)
print(
    "Unique master_ccl_id:",
    ctrp_model_crosswalk["master_ccl_id"].nunique(),
)
print(
    "Anchor coverage:",
    f"{100 * len(ctrp_model_crosswalk) / len(anchor_model_ids):.2f}%",
)

Final CTRP mapped models: 566
Unique ModelID: 566
Unique master_ccl_id: 566
Anchor coverage: 79.38%


In [16]:
# =============================================================================
# Summarize cross-resource anchor-model coverage
# =============================================================================

prism_any_screen_ids = set(prism_models["depmap_id"])
prism_any_screen_overlap = (
    prism_any_screen_ids & anchor_model_ids
)

model_coverage_summary = pd.DataFrame(
    [
        {
            "resource": "GDSC",
            "anchor_models": len(anchor_model_ids),
            "response_covered_anchor_models": len(gdsc_anchor_overlap),
            "anchor_coverage_pct": (
                100 * len(gdsc_anchor_overlap) / len(anchor_model_ids)
            ),
        },
        {
            "resource": "CTRP",
            "anchor_models": len(anchor_model_ids),
            "response_covered_anchor_models": len(ctrp_model_crosswalk),
            "anchor_coverage_pct": (
                100 * len(ctrp_model_crosswalk) / len(anchor_model_ids)
            ),
        },
        {
            "resource": "PRISM any screen",
            "anchor_models": len(anchor_model_ids),
            "response_covered_anchor_models": len(prism_any_screen_overlap),
            "anchor_coverage_pct": (
                100 * len(prism_any_screen_overlap) / len(anchor_model_ids)
            ),
        },
    ]
)

model_coverage_summary

,resource,anchor_models,response_covered_anchor_models,anchor_coverage_pct
0,GDSC,713,713,100.000000
1,CTRP,713,566,79.382889
2,PRISM any screen,713,341,47.826087


In [17]:
# =============================================================================
# Reproduce frozen Phase 4 cell-line consensus scores
# =============================================================================

harmonized_expression = pd.read_parquet(
    upstream_artifact_paths["phase3.303.harmonized_expression"]
)

shared_gene_universe = pd.read_csv(
    upstream_artifact_paths[
        "phase4.400.cross_system_shared_gene_universe"
    ]
)

consensus_gene_weights = pd.read_csv(
    upstream_artifact_paths[
        "phase4.401.consensus_transcriptomic_gene_weights"
    ]
)

program_columns = [
    column
    for column in consensus_cellline_scores.columns
    if column != "ModelID"
]

consensus_weight_matrix = (
    consensus_gene_weights
    .pivot(
        index="gene_symbol",
        columns="consensus_program_id",
        values="consensus_weight",
    )
    .reindex(shared_gene_universe["gene_symbol"])
    [program_columns]
)

cell_line_projection_columns = (
    shared_gene_universe["cell_line_gene_id"].tolist()
)

expression_matrix = (
    harmonized_expression[cell_line_projection_columns]
    .to_numpy(dtype=np.float64)
)

gene_means = expression_matrix.mean(axis=0, keepdims=True)
gene_stds = expression_matrix.std(axis=0, ddof=1, keepdims=True)

standardized_expression = (
    expression_matrix - gene_means
) / gene_stds

projected_scores = (
    standardized_expression
    @ consensus_weight_matrix.to_numpy(dtype=np.float64)
)

score_means = projected_scores.mean(axis=0, keepdims=True)
score_stds = projected_scores.std(axis=0, ddof=1, keepdims=True)

reproduced_score_values = (
    projected_scores - score_means
) / score_stds

reproduced_scores = pd.DataFrame(
    reproduced_score_values,
    columns=program_columns,
)

reproduced_scores.insert(
    0,
    "ModelID",
    harmonized_expression["ModelID"].to_numpy(),
)

score_comparison = reproduced_scores.merge(
    consensus_cellline_scores,
    on="ModelID",
    how="inner",
    validate="one_to_one",
    suffixes=("_reproduced", "_frozen"),
)

max_absolute_differences = {
    program: (
        score_comparison[f"{program}_reproduced"]
        - score_comparison[f"{program}_frozen"]
    )
    .abs()
    .max()
    for program in program_columns
}

print("Models compared:", len(score_comparison))
print("Maximum absolute differences:")
for program, difference in max_absolute_differences.items():
    print(f"  {program}: {difference:.3e}")

Models compared: 713
Maximum absolute differences:
  CONSENSUS_TX_01: 2.665e-15
  CONSENSUS_TX_02: 1.776e-15
  CONSENSUS_TX_03: 2.220e-15


In [18]:
# =============================================================================
# Quantify potential model expansion beyond the frozen anchor cohort
# =============================================================================

ctrp_response_cell_ids = set(response_master_ccl_ids)
ctrp_anchor_response_cell_ids = set(
    ctrp_model_crosswalk["master_ccl_id"]
)

prism_response_model_ids = set(prism_models["depmap_id"])
prism_anchor_response_model_ids = (
    prism_response_model_ids & anchor_model_ids
)

potential_expansion_summary = pd.DataFrame(
    [
        {
            "resource": "CTRP",
            "response_models_or_cells": len(ctrp_response_cell_ids),
            "mapped_anchor_models": len(ctrp_anchor_response_cell_ids),
            "outside_anchor_or_unresolved": (
                len(ctrp_response_cell_ids)
                - len(ctrp_anchor_response_cell_ids)
            ),
        },
        {
            "resource": "PRISM",
            "response_models_or_cells": len(prism_response_model_ids),
            "mapped_anchor_models": len(prism_anchor_response_model_ids),
            "outside_anchor_or_unresolved": (
                len(prism_response_model_ids)
                - len(prism_anchor_response_model_ids)
            ),
        },
    ]
)

potential_expansion_summary

,resource,response_models_or_cells,mapped_anchor_models,outside_anchor_or_unresolved
0,CTRP,887,566,321
1,PRISM,480,341,139


In [19]:
# =============================================================================
# Quantify expression-eligible external models in CTRP and PRISM
# =============================================================================

depmap_models = pd.read_csv(
    Paths.depmap / "Model.csv",
    usecols=["ModelID", "CellLineName"],
)

depmap_models["name_key"] = (
    depmap_models["CellLineName"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

depmap_key_counts = depmap_models["name_key"].value_counts()
ctrp_key_counts_full = ctrp_cell_lines["name_key"].value_counts()

shared_full_keys = (
    set(depmap_key_counts.index)
    & set(ctrp_key_counts_full.index)
)

unambiguous_full_keys = {
    key
    for key in shared_full_keys
    if depmap_key_counts[key] == 1
    and ctrp_key_counts_full[key] == 1
}

ctrp_depmap_crosswalk = (
    ctrp_cell_lines.loc[
        ctrp_cell_lines["name_key"].isin(unambiguous_full_keys),
        ["master_ccl_id", "ccl_name", "name_key"],
    ]
    .merge(
        depmap_models.loc[
            depmap_models["name_key"].isin(unambiguous_full_keys),
            ["ModelID", "CellLineName", "name_key"],
        ],
        on="name_key",
        how="inner",
        validate="one_to_one",
    )
)

ctrp_external_model_ids = set(
    ctrp_depmap_crosswalk.loc[
        ctrp_depmap_crosswalk["master_ccl_id"].isin(
            response_master_ccl_ids
        ),
        "ModelID",
    ]
) - anchor_model_ids

prism_external_model_ids = (
    prism_response_model_ids - anchor_model_ids
)

depmap_expression_model_ids = set(
    pd.read_csv(
        Paths.depmap / "OmicsExpressionProteinCodingGenesTPMLogp1.csv",
        usecols=["Unnamed: 0"],
    )["Unnamed: 0"]
)

external_expression_coverage = pd.DataFrame(
    [
        {
            "resource": "CTRP",
            "deterministic_external_models": len(
                ctrp_external_model_ids
            ),
            "with_depmap_expression": len(
                ctrp_external_model_ids
                & depmap_expression_model_ids
            ),
        },
        {
            "resource": "PRISM",
            "deterministic_external_models": len(
                prism_external_model_ids
            ),
            "with_depmap_expression": len(
                prism_external_model_ids
                & depmap_expression_model_ids
            ),
        },
    ]
)

external_expression_coverage["expression_coverage_pct"] = (
    100
    * external_expression_coverage["with_depmap_expression"]
    / external_expression_coverage["deterministic_external_models"]
)

external_expression_coverage

,resource,deterministic_external_models,with_depmap_expression,expression_coverage_pct
0,CTRP,272,256,94.117647
1,PRISM,139,136,97.841727


In [20]:
# =============================================================================
# Quantify unique expression-eligible external model expansion
# =============================================================================

ctrp_external_expression_ids = (
    ctrp_external_model_ids
    & depmap_expression_model_ids
)

prism_external_expression_ids = (
    prism_external_model_ids
    & depmap_expression_model_ids
)

shared_external_expression_ids = (
    ctrp_external_expression_ids
    & prism_external_expression_ids
)

external_expression_union_ids = (
    ctrp_external_expression_ids
    | prism_external_expression_ids
)

print(
    "CTRP expression-eligible external models:",
    len(ctrp_external_expression_ids),
)
print(
    "PRISM expression-eligible external models:",
    len(prism_external_expression_ids),
)
print(
    "Shared external models across CTRP and PRISM:",
    len(shared_external_expression_ids),
)
print(
    "Unique external models across both screens:",
    len(external_expression_union_ids),
)
print(
    "Total scoreable models after expansion:",
    len(anchor_model_ids | external_expression_union_ids),
)
print(
    "Increase over frozen 713-model anchor:",
    f"{100 * len(external_expression_union_ids) / len(anchor_model_ids):.2f}%",
)

CTRP expression-eligible external models: 256
PRISM expression-eligible external models: 136
Shared external models across CTRP and PRISM: 124
Unique external models across both screens: 268
Total scoreable models after expansion: 981
Increase over frozen 713-model anchor: 37.59%


In [21]:
# =============================================================================
# Project consensus scores to expression-eligible external models
# =============================================================================

external_expression = pd.read_csv(
    Paths.depmap / "OmicsExpressionProteinCodingGenesTPMLogp1.csv",
    usecols=["Unnamed: 0", *cell_line_projection_columns],
).rename(columns={"Unnamed: 0": "ModelID"})

external_expression = (
    external_expression.loc[
        external_expression["ModelID"].isin(
            external_expression_union_ids
        )
    ]
    .copy()
)

external_expression_matrix = (
    external_expression[cell_line_projection_columns]
    .to_numpy(dtype=np.float64)
)

external_standardized_expression = (
    external_expression_matrix - gene_means
) / gene_stds

external_projected_scores = (
    external_standardized_expression
    @ consensus_weight_matrix.to_numpy(dtype=np.float64)
)

external_score_values = (
    external_projected_scores - score_means
) / score_stds

external_consensus_scores = pd.DataFrame(
    external_score_values,
    columns=program_columns,
)

external_consensus_scores.insert(
    0,
    "ModelID",
    external_expression["ModelID"].to_numpy(),
)

print(
    "Expected external models:",
    len(external_expression_union_ids),
)
print(
    "Projected external models:",
    len(external_consensus_scores),
)
print(
    "Missing expected models:",
    len(
        external_expression_union_ids
        - set(external_consensus_scores["ModelID"])
    ),
)
print(
    "Missing projected scores:",
    int(
        external_consensus_scores[
            program_columns
        ].isna().sum().sum()
    ),
)

Expected external models: 268
Projected external models: 268
Missing expected models: 0
Missing projected scores: 0


In [22]:
# =============================================================================
# Build expanded Phase 6 score universe with lineage metadata
# =============================================================================

external_model_metadata = pd.read_csv(
    Paths.depmap / "Model.csv",
    usecols=["ModelID", "OncotreeLineage"],
)

external_model_metadata = external_model_metadata.loc[
    external_model_metadata["ModelID"].isin(
        external_expression_union_ids
    )
].copy()

anchor_score_universe = (
    phase6_anchor_cohort[
        ["ModelID", "OncotreeLineage", *program_columns]
    ]
    .assign(score_origin="frozen_phase4")
)

external_score_universe = (
    external_consensus_scores
    .merge(
        external_model_metadata,
        on="ModelID",
        how="left",
        validate="one_to_one",
    )
    .assign(score_origin="projected_from_frozen_phase4")
)

phase6_score_universe = pd.concat(
    [
        anchor_score_universe,
        external_score_universe[
            ["ModelID", "OncotreeLineage", *program_columns, "score_origin"]
        ],
    ],
    ignore_index=True,
)

print("Phase 6 score universe:", len(phase6_score_universe))
print(
    "Unique ModelID:",
    phase6_score_universe["ModelID"].nunique(),
)
print(
    "Frozen Phase 4 models:",
    (phase6_score_universe["score_origin"] == "frozen_phase4").sum(),
)
print(
    "Projected external models:",
    (
        phase6_score_universe["score_origin"]
        == "projected_from_frozen_phase4"
    ).sum(),
)
print(
    "Models missing lineage:",
    phase6_score_universe["OncotreeLineage"].isna().sum(),
)
print(
    "External models missing lineage:",
    external_score_universe["OncotreeLineage"].isna().sum(),
)

Phase 6 score universe: 981
Unique ModelID: 981
Frozen Phase 4 models: 713
Projected external models: 268
Models missing lineage: 0
External models missing lineage: 0


In [23]:
# =============================================================================
# Characterize scoreable model coverage by resource and lineage
# =============================================================================

phase6_scoreable_ids = set(
    phase6_score_universe["ModelID"]
)

gdsc_scoreable_ids = set(
    phase6_anchor_cohort.loc[
        phase6_anchor_cohort["SangerModelID"].isin(
            gdsc_response_ids
        ),
        "ModelID",
    ]
)

ctrp_scoreable_ids = set(
    ctrp_depmap_crosswalk.loc[
        ctrp_depmap_crosswalk["master_ccl_id"].isin(
            response_master_ccl_ids
        ),
        "ModelID",
    ]
) & phase6_scoreable_ids

prism_scoreable_ids = (
    prism_response_model_ids
    & phase6_scoreable_ids
)

scoreable_resource_models = {
    "GDSC": gdsc_scoreable_ids,
    "CTRP": ctrp_scoreable_ids,
    "PRISM": prism_scoreable_ids,
}

resource_coverage_rows = []
lineage_coverage_rows = []

for resource, model_ids in scoreable_resource_models.items():
    resource_models = phase6_score_universe.loc[
        phase6_score_universe["ModelID"].isin(model_ids),
        ["ModelID", "OncotreeLineage"],
    ]

    resource_coverage_rows.append(
        {
            "resource": resource,
            "scoreable_models": len(resource_models),
            "represented_lineages": (
                resource_models["OncotreeLineage"].nunique()
            ),
        }
    )

    lineage_counts = (
        resource_models["OncotreeLineage"]
        .value_counts()
    )

    for lineage, count in lineage_counts.items():
        lineage_coverage_rows.append(
            {
                "resource": resource,
                "OncotreeLineage": lineage,
                "models": count,
            }
        )

resource_scoreable_coverage = pd.DataFrame(
    resource_coverage_rows
)

lineage_scoreable_coverage = (
    pd.DataFrame(lineage_coverage_rows)
    .pivot(
        index="OncotreeLineage",
        columns="resource",
        values="models",
    )
    .fillna(0)
    .astype(int)
)

display(resource_scoreable_coverage)
display(lineage_scoreable_coverage)

,resource,scoreable_models,represented_lineages
0,GDSC,713,27
1,CTRP,821,25
2,PRISM,477,22


resource,CTRP,GDSC,PRISM
OncotreeLineage,,,
Adrenal Gland,0,1,0
Ampulla of Vater,2,0,1
Biliary Tract,6,2,6
Bladder/Urinary Tract,24,16,23
Bone,15,17,12
Bowel,48,43,27
Breast,40,47,22
CNS/Brain,49,37,34
Cervix,1,11,0


In [24]:
# =============================================================================
# Reconcile CTRP anchor and expanded deterministic crosswalks
# =============================================================================

ctrp_anchor_response_ids = set(
    ctrp_model_crosswalk["ModelID"]
)

ctrp_full_response_ids = set(
    ctrp_depmap_crosswalk.loc[
        ctrp_depmap_crosswalk["master_ccl_id"].isin(
            response_master_ccl_ids
        ),
        "ModelID",
    ]
)

lost_anchor_ids = (
    ctrp_anchor_response_ids
    - ctrp_full_response_ids
)

external_full_ids = (
    ctrp_full_response_ids
    - anchor_model_ids
)

print(
    "Anchor CTRP models from original crosswalk:",
    len(ctrp_anchor_response_ids),
)
print(
    "Anchor CTRP models retained in full-DepMap crosswalk:",
    len(ctrp_full_response_ids & anchor_model_ids),
)
print(
    "Anchor models lost after full-DepMap ambiguity check:",
    len(lost_anchor_ids),
)
print(
    "External CTRP models in full-DepMap crosswalk:",
    len(external_full_ids),
)

if lost_anchor_ids:
    lost_keys = set(
        ctrp_anchor_crosswalk.loc[
            ctrp_anchor_crosswalk["ModelID"].isin(lost_anchor_ids),
            "name_key",
        ]
    )

    print("\nLost anchor mapping:")
    display(
        ctrp_anchor_crosswalk.loc[
            ctrp_anchor_crosswalk["ModelID"].isin(lost_anchor_ids),
            [
                "master_ccl_id",
                "ccl_name",
                "name_key",
                "ModelID",
                "CellLineName",
            ],
        ]
    )

    print("\nDepMap models sharing the affected normalized key:")
    display(
        depmap_models.loc[
            depmap_models["name_key"].isin(lost_keys),
            [
                "ModelID",
                "CellLineName",
                "name_key",
            ],
        ].sort_values(["name_key", "ModelID"])
    )

Anchor CTRP models from original crosswalk: 566
Anchor CTRP models retained in full-DepMap crosswalk: 565
Anchor models lost after full-DepMap ambiguity check: 1
External CTRP models in full-DepMap crosswalk: 272

Lost anchor mapping:


,master_ccl_id,ccl_name,name_key,ModelID,CellLineName
244,538,KMH2,KMH2,ACH-000815,KM-H2



DepMap models sharing the affected normalized key:


,ModelID,CellLineName,name_key
811,ACH-000815,KM-H2,KMH2
1868,ACH-002397,KMH-2,KMH2


In [25]:
# =============================================================================
# Finalize globally unambiguous CTRP Phase 6 model crosswalk
# =============================================================================

ctrp_phase6_crosswalk = (
    ctrp_depmap_crosswalk.loc[
        ctrp_depmap_crosswalk["master_ccl_id"].isin(
            response_master_ccl_ids
        )
        & ctrp_depmap_crosswalk["ModelID"].isin(
            phase6_scoreable_ids
        ),
        [
            "master_ccl_id",
            "ccl_name",
            "ModelID",
            "CellLineName",
        ],
    ]
    .assign(
        mapping_method="exact_normalized_name_globally_unambiguous"
    )
    .sort_values("ModelID")
    .reset_index(drop=True)
)

ctrp_phase6_anchor_ids = (
    set(ctrp_phase6_crosswalk["ModelID"])
    & anchor_model_ids
)

ctrp_phase6_external_ids = (
    set(ctrp_phase6_crosswalk["ModelID"])
    - anchor_model_ids
)

print(
    "Final CTRP Phase 6 scoreable models:",
    len(ctrp_phase6_crosswalk),
)
print(
    "Anchor models:",
    len(ctrp_phase6_anchor_ids),
)
print(
    "Projected external models:",
    len(ctrp_phase6_external_ids),
)
print(
    "Unique ModelID:",
    ctrp_phase6_crosswalk["ModelID"].nunique(),
)
print(
    "Unique master_ccl_id:",
    ctrp_phase6_crosswalk["master_ccl_id"].nunique(),
)
print(
    "Excluded globally ambiguous CTRP cell lines:",
    len(lost_anchor_ids),
)

Final CTRP Phase 6 scoreable models: 821
Anchor models: 565
Projected external models: 256
Unique ModelID: 821
Unique master_ccl_id: 821
Excluded globally ambiguous CTRP cell lines: 1


In [26]:
# =============================================================================
# Characterize native drug-level model coverage across pharmacogenomic resources
# =============================================================================

gdsc_drug_models = (
    pd.read_excel(
        pharmacogenomic_input_paths[("gdsc", "response")],
        usecols=["SANGER_MODEL_ID", "DRUG_ID"],
    )
    .merge(
        phase6_anchor_cohort[
            ["ModelID", "SangerModelID"]
        ],
        left_on="SANGER_MODEL_ID",
        right_on="SangerModelID",
        how="inner",
        validate="many_to_one",
    )
    [["DRUG_ID", "ModelID"]]
    .drop_duplicates()
)

with zipfile.ZipFile(
    pharmacogenomic_input_paths[("ctrp", "response_archive")]
) as ctrp_archive:
    ctrp_drug_models = pd.read_csv(
        ctrp_archive.open("v20.data.curves_post_qc.txt"),
        sep="\t",
        usecols=["experiment_id", "master_cpd_id"],
    )

ctrp_drug_models = (
    ctrp_drug_models
    .merge(
        ctrp_experiment_map,
        on="experiment_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        ctrp_phase6_crosswalk[
            ["master_ccl_id", "ModelID"]
        ],
        on="master_ccl_id",
        how="inner",
        validate="many_to_one",
    )
    [["master_cpd_id", "ModelID"]]
    .drop_duplicates()
)

prism_drug_models = (
    pd.read_csv(
        pharmacogenomic_input_paths[("prism", "response")],
        usecols=["broad_id", "depmap_id", "screen_id"],
    )
    .loc[
        lambda data: data["depmap_id"].isin(
            phase6_scoreable_ids
        )
    ]
    [["screen_id", "broad_id", "depmap_id"]]
    .drop_duplicates()
)

drug_coverage_rows = []

for resource, data, drug_column, model_column in (
    ("GDSC", gdsc_drug_models, "DRUG_ID", "ModelID"),
    ("CTRP", ctrp_drug_models, "master_cpd_id", "ModelID"),
):
    model_counts = data.groupby(drug_column)[model_column].nunique()

    drug_coverage_rows.append(
        {
            "resource": resource,
            "native_drugs": len(model_counts),
            "min_models_per_drug": int(model_counts.min()),
            "median_models_per_drug": float(model_counts.median()),
            "max_models_per_drug": int(model_counts.max()),
        }
    )

for screen_id, screen_data in prism_drug_models.groupby("screen_id"):
    model_counts = (
        screen_data.groupby("broad_id")["depmap_id"].nunique()
    )

    drug_coverage_rows.append(
        {
            "resource": f"PRISM {screen_id}",
            "native_drugs": len(model_counts),
            "min_models_per_drug": int(model_counts.min()),
            "median_models_per_drug": float(model_counts.median()),
            "max_models_per_drug": int(model_counts.max()),
        }
    )

native_drug_coverage = pd.DataFrame(drug_coverage_rows)

native_drug_coverage

,resource,native_drugs,min_models_per_drug,median_models_per_drug,max_models_per_drug
0,GDSC,295,136,656.0,713
1,CTRP,545,83,750.0,807
2,PRISM HTS002,1396,27,436.5,474
3,PRISM MTS005,2,407,408.0,409
4,PRISM MTS006,73,401,453.0,470
5,PRISM MTS010,147,358,430.0,460


In [27]:
# =============================================================================
# Construct native compound identity catalogs
# =============================================================================

gdsc_compounds = (
    pd.read_excel(
        pharmacogenomic_input_paths[("gdsc", "response")],
        usecols=["DRUG_ID", "DRUG_NAME"],
    )
    .drop_duplicates()
)

with zipfile.ZipFile(
    pharmacogenomic_input_paths[("ctrp", "response_archive")]
) as ctrp_archive:
    ctrp_compounds = pd.read_csv(
        ctrp_archive.open("v20.meta.per_compound.txt"),
        sep="\t",
        usecols=["master_cpd_id", "cpd_name"],
    )

prism_compounds = (
    pd.read_csv(
        pharmacogenomic_input_paths[("prism", "response")],
        usecols=["broad_id", "name"],
    )
    .drop_duplicates()
)

for data, name_column in (
    (gdsc_compounds, "DRUG_NAME"),
    (ctrp_compounds, "cpd_name"),
    (prism_compounds, "name"),
):
    data["drug_name_key"] = (
        data[name_column]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"[^A-Z0-9]", "", regex=True)
    )

compound_identity_summary = pd.DataFrame(
    [
        {
            "resource": "GDSC",
            "native_ids": gdsc_compounds["DRUG_ID"].nunique(),
            "native_names": gdsc_compounds["DRUG_NAME"].nunique(),
            "normalized_name_keys": gdsc_compounds["drug_name_key"].nunique(),
            "normalized_key_collisions": (
                gdsc_compounds.groupby("drug_name_key")["DRUG_ID"]
                .nunique()
                .gt(1)
                .sum()
            ),
        },
        {
            "resource": "CTRP",
            "native_ids": ctrp_compounds["master_cpd_id"].nunique(),
            "native_names": ctrp_compounds["cpd_name"].nunique(),
            "normalized_name_keys": ctrp_compounds["drug_name_key"].nunique(),
            "normalized_key_collisions": (
                ctrp_compounds.groupby("drug_name_key")["master_cpd_id"]
                .nunique()
                .gt(1)
                .sum()
            ),
        },
        {
            "resource": "PRISM",
            "native_ids": prism_compounds["broad_id"].nunique(),
            "native_names": prism_compounds["name"].nunique(),
            "normalized_name_keys": prism_compounds["drug_name_key"].nunique(),
            "normalized_key_collisions": (
                prism_compounds.groupby("drug_name_key")["broad_id"]
                .nunique()
                .gt(1)
                .sum()
            ),
        },
    ]
)

compound_identity_summary

,resource,native_ids,native_names,normalized_name_keys,normalized_key_collisions
0,GDSC,295,286,286,9
1,CTRP,545,545,545,0
2,PRISM,1502,1448,1446,56


In [28]:
# =============================================================================
# Quantify unambiguous exact-name compound overlap across resources
# =============================================================================

gdsc_unique_keys = {
    key
    for key, count in (
        gdsc_compounds.groupby("drug_name_key")["DRUG_ID"]
        .nunique()
        .items()
    )
    if count == 1
}

ctrp_unique_keys = {
    key
    for key, count in (
        ctrp_compounds.groupby("drug_name_key")["master_cpd_id"]
        .nunique()
        .items()
    )
    if count == 1
}

prism_unique_keys = {
    key
    for key, count in (
        prism_compounds.groupby("drug_name_key")["broad_id"]
        .nunique()
        .items()
    )
    if count == 1
}

gdsc_ctrp_keys = (
    gdsc_unique_keys
    & ctrp_unique_keys
)

gdsc_prism_keys = (
    gdsc_unique_keys
    & prism_unique_keys
)

ctrp_prism_keys = (
    ctrp_unique_keys
    & prism_unique_keys
)

three_way_keys = (
    gdsc_unique_keys
    & ctrp_unique_keys
    & prism_unique_keys
)

exact_name_overlap_summary = pd.DataFrame(
    [
        {
            "comparison": "GDSC–CTRP",
            "unambiguous_exact_name_matches": len(gdsc_ctrp_keys),
        },
        {
            "comparison": "GDSC–PRISM",
            "unambiguous_exact_name_matches": len(gdsc_prism_keys),
        },
        {
            "comparison": "CTRP–PRISM",
            "unambiguous_exact_name_matches": len(ctrp_prism_keys),
        },
        {
            "comparison": "GDSC–CTRP–PRISM",
            "unambiguous_exact_name_matches": len(three_way_keys),
        },
    ]
)

exact_name_overlap_summary

,comparison,unambiguous_exact_name_matches
0,GDSC–CTRP,68
1,GDSC–PRISM,87
2,CTRP–PRISM,151
3,GDSC–CTRP–PRISM,39


In [29]:
# =============================================================================
# Construct unambiguous exact-name cross-resource compound crosswalk
# =============================================================================

gdsc_exact = (
    gdsc_compounds.loc[
        gdsc_compounds["drug_name_key"].isin(gdsc_unique_keys),
        ["drug_name_key", "DRUG_ID", "DRUG_NAME"],
    ]
    .rename(
        columns={
            "DRUG_ID": "gdsc_drug_id",
            "DRUG_NAME": "gdsc_drug_name",
        }
    )
)

ctrp_exact = (
    ctrp_compounds.loc[
        ctrp_compounds["drug_name_key"].isin(ctrp_unique_keys),
        ["drug_name_key", "master_cpd_id", "cpd_name"],
    ]
    .rename(
        columns={
            "master_cpd_id": "ctrp_master_cpd_id",
            "cpd_name": "ctrp_drug_name",
        }
    )
)

prism_exact = (
    prism_compounds.loc[
        prism_compounds["drug_name_key"].isin(prism_unique_keys),
        ["drug_name_key", "broad_id", "name"],
    ]
    .rename(
        columns={
            "broad_id": "prism_broad_id",
            "name": "prism_drug_name",
        }
    )
)

exact_drug_crosswalk = (
    gdsc_exact
    .merge(
        ctrp_exact,
        on="drug_name_key",
        how="outer",
        validate="one_to_one",
    )
    .merge(
        prism_exact,
        on="drug_name_key",
        how="outer",
        validate="one_to_one",
    )
)

exact_drug_crosswalk["resources_present"] = (
    exact_drug_crosswalk[
        [
            "gdsc_drug_id",
            "ctrp_master_cpd_id",
            "prism_broad_id",
        ]
    ]
    .notna()
    .sum(axis=1)
)

cross_resource_exact_drugs = (
    exact_drug_crosswalk.loc[
        exact_drug_crosswalk["resources_present"] >= 2
    ]
    .sort_values(
        ["resources_present", "drug_name_key"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print(
    "Exact-name compounds present in >=2 resources:",
    len(cross_resource_exact_drugs),
)
print(
    "Present in all 3 resources:",
    (cross_resource_exact_drugs["resources_present"] == 3).sum(),
)
print(
    "Present in exactly 2 resources:",
    (cross_resource_exact_drugs["resources_present"] == 2).sum(),
)

display(cross_resource_exact_drugs.head())

Exact-name compounds present in >=2 resources: 228
Present in all 3 resources: 39
Present in exactly 2 resources: 189


,drug_name_key,gdsc_drug_id,gdsc_drug_name,ctrp_master_cpd_id,ctrp_drug_name,prism_broad_id,prism_drug_name,resources_present
0,ABT737,1910.0,ABT737,411738.0,ABT-737,BRD-K56301217-001-07-4,ABT-737,3
1,ALISERTIB,1051.0,Alisertib,636711.0,alisertib,BRD-K75295174-001-05-0,alisertib,3
2,AXITINIB,1021.0,Axitinib,348990.0,axitinib,BRD-K29905972-001-06-3,axitinib,3
3,AZD4547,1786.0,AZD4547,660325.0,AZD4547,BRD-K28392481-001-05-1,AZD4547,3
4,AZD6482,2169.0,AZD6482,639390.0,AZD6482,BRD-K58772419-001-07-0,AZD6482,3


In [30]:
# =============================================================================
# Annotate exact cross-resource compounds with model coverage
# =============================================================================

gdsc_model_counts = (
    gdsc_drug_models
    .groupby("DRUG_ID")["ModelID"]
    .nunique()
    .rename("gdsc_models")
)

ctrp_model_counts = (
    ctrp_drug_models
    .groupby("master_cpd_id")["ModelID"]
    .nunique()
    .rename("ctrp_models")
)

prism_model_counts = (
    prism_drug_models
    .groupby(["broad_id", "screen_id"])["depmap_id"]
    .nunique()
    .unstack(fill_value=0)
    .rename(
        columns=lambda screen: f"prism_{screen}_models"
    )
)

cross_resource_drug_coverage = (
    cross_resource_exact_drugs
    .merge(
        gdsc_model_counts,
        left_on="gdsc_drug_id",
        right_index=True,
        how="left",
        validate="many_to_one",
    )
    .merge(
        ctrp_model_counts,
        left_on="ctrp_master_cpd_id",
        right_index=True,
        how="left",
        validate="many_to_one",
    )
    .merge(
        prism_model_counts,
        left_on="prism_broad_id",
        right_index=True,
        how="left",
        validate="many_to_one",
    )
)

coverage_columns = [
    column
    for column in cross_resource_drug_coverage.columns
    if column.endswith("_models")
]

cross_resource_drug_coverage[
    coverage_columns
] = (
    cross_resource_drug_coverage[
        coverage_columns
    ]
    .fillna(0)
    .astype(int)
)

print(
    "Cross-resource exact compounds:",
    len(cross_resource_drug_coverage),
)

print("\nPRISM screen availability among exact cross-resource compounds:")
for column in [
    column
    for column in coverage_columns
    if column.startswith("prism_")
]:
    print(
        f"  {column}:",
        (cross_resource_drug_coverage[column] > 0).sum(),
    )

display(
    cross_resource_drug_coverage[
        [
            "drug_name_key",
            "resources_present",
            *coverage_columns,
        ]
    ].head(10)
)

Cross-resource exact compounds: 228

PRISM screen availability among exact cross-resource compounds:
  prism_HTS002_models: 192
  prism_MTS005_models: 0
  prism_MTS006_models: 10
  prism_MTS010_models: 37


,drug_name_key,resources_present,gdsc_models,ctrp_models,prism_HTS002_models,prism_MTS005_models,prism_MTS006_models,prism_MTS010_models
0,ABT737,3,703,708,452,0,0,0
1,ALISERTIB,3,697,774,471,0,0,0
2,AXITINIB,3,706,773,437,0,0,430
3,AZD4547,3,706,702,344,0,0,0
4,AZD6482,3,175,785,473,0,0,0
5,AZD7762,3,707,788,466,0,0,0
6,AZD8055,3,579,794,450,0,0,0
7,BI2536,3,675,790,465,0,0,0
8,BIBR1532,3,701,773,467,0,0,0
9,BMS345541,3,658,754,453,0,0,0


In [31]:
# =============================================================================
# Characterize PRISM screen patterns among cross-resource compounds
# =============================================================================

prism_screen_columns = [
    "prism_HTS002_models",
    "prism_MTS005_models",
    "prism_MTS006_models",
    "prism_MTS010_models",
]

prism_cross_resource_compounds = (
    cross_resource_drug_coverage.loc[
        cross_resource_drug_coverage["prism_broad_id"].notna()
    ]
    .copy()
)

prism_cross_resource_compounds["screen_pattern"] = (
    prism_cross_resource_compounds[
        prism_screen_columns
    ]
    .gt(0)
    .apply(
        lambda row: "+".join(
            column.removeprefix("prism_").removesuffix("_models")
            for column, present in row.items()
            if present
        ),
        axis=1,
    )
)

prism_screen_pattern_summary = (
    prism_cross_resource_compounds["screen_pattern"]
    .value_counts()
    .rename_axis("screen_pattern")
    .reset_index(name="compounds")
)

display(prism_screen_pattern_summary)

print(
    "Cross-resource compounds represented in PRISM:",
    len(prism_cross_resource_compounds),
)
print(
    "Compounds with MTS010:",
    (
        prism_cross_resource_compounds["prism_MTS010_models"] > 0
    ).sum(),
)
print(
    "MTS010 compounds also represented in HTS002:",
    (
        (prism_cross_resource_compounds["prism_MTS010_models"] > 0)
        & (prism_cross_resource_compounds["prism_HTS002_models"] > 0)
    ).sum(),
)
print(
    "MTS006 compounds also represented in HTS002:",
    (
        (prism_cross_resource_compounds["prism_MTS006_models"] > 0)
        & (prism_cross_resource_compounds["prism_HTS002_models"] > 0)
    ).sum(),
)

,screen_pattern,compounds
0,HTS002,156
1,HTS002+MTS010,33
2,MTS006,4
3,MTS006+MTS010,3
4,HTS002+MTS006,2
5,HTS002+MTS006+MTS010,1


Cross-resource compounds represented in PRISM: 199
Compounds with MTS010: 37
MTS010 compounds also represented in HTS002: 34
MTS006 compounds also represented in HTS002: 3


In [32]:
# =============================================================================
# Define prospective primary PRISM screen rule
# =============================================================================

def assign_primary_prism_screen(row):
    if row["prism_MTS010_models"] > 0:
        return "MTS010"

    available_nonredo_screens = [
        screen
        for screen in ("HTS002", "MTS005", "MTS006")
        if row[f"prism_{screen}_models"] > 0
    ]

    if len(available_nonredo_screens) == 1:
        return available_nonredo_screens[0]

    if len(available_nonredo_screens) > 1:
        return "AMBIGUOUS_NONREDO"

    return pd.NA


prism_cross_resource_compounds["primary_prism_screen"] = (
    prism_cross_resource_compounds.apply(
        assign_primary_prism_screen,
        axis=1,
    )
)

primary_prism_screen_summary = (
    prism_cross_resource_compounds["primary_prism_screen"]
    .value_counts(dropna=False)
    .rename_axis("primary_prism_screen")
    .reset_index(name="compounds")
)

display(primary_prism_screen_summary)

print(
    "Compounds eligible for a unique primary PRISM screen:",
    (
        prism_cross_resource_compounds["primary_prism_screen"]
        != "AMBIGUOUS_NONREDO"
    ).sum(),
)
print(
    "Compounds retained for screen-specific sensitivity only:",
    (
        prism_cross_resource_compounds["primary_prism_screen"]
        == "AMBIGUOUS_NONREDO"
    ).sum(),
)

,primary_prism_screen,compounds
0,HTS002,156
1,MTS010,37
2,MTS006,4
3,AMBIGUOUS_NONREDO,2


Compounds eligible for a unique primary PRISM screen: 197
Compounds retained for screen-specific sensitivity only: 2


In [33]:
# =============================================================================
# Define prospective primary pharmacogenomic response metrics
# =============================================================================

PRIMARY_RESPONSE_METRICS = {
    "GDSC": {
        "metric": "LN_IC50",
        "higher_is": "resistance_like",
        "role": "developmental_internal",
    },
    "CTRP": {
        "metric": "area_under_curve",
        "higher_is": "resistance_like",
        "role": "external_replication",
    },
    "PRISM": {
        "metric": "auc",
        "higher_is": "resistance_like",
        "role": "external_replication",
    },
}

primary_response_metric_summary = pd.DataFrame(
    [
        {
            "resource": resource,
            **specification,
        }
        for resource, specification
        in PRIMARY_RESPONSE_METRICS.items()
    ]
)

primary_response_metric_summary

,resource,metric,higher_is,role
0,GDSC,LN_IC50,resistance_like,developmental_internal
1,CTRP,area_under_curve,resistance_like,external_replication
2,PRISM,auc,resistance_like,external_replication


In [34]:
# =============================================================================
# Characterize repeated CTRP AUC measurements in the Phase 6 universe
# =============================================================================

with zipfile.ZipFile(
    pharmacogenomic_input_paths[("ctrp", "response_archive")]
) as ctrp_archive:
    ctrp_auc_response = pd.read_csv(
        ctrp_archive.open("v20.data.curves_post_qc.txt"),
        sep="\t",
        usecols=[
            "experiment_id",
            "master_cpd_id",
            "area_under_curve",
        ],
    )

ctrp_auc_phase6 = (
    ctrp_auc_response
    .merge(
        ctrp_experiment_map,
        on="experiment_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        ctrp_phase6_crosswalk[
            ["master_ccl_id", "ModelID"]
        ],
        on="master_ccl_id",
        how="inner",
        validate="many_to_one",
    )
)

ctrp_pair_summary = (
    ctrp_auc_phase6
    .groupby(["ModelID", "master_cpd_id"])["area_under_curve"]
    .agg(
        experiments="size",
        auc_min="min",
        auc_max="max",
    )
    .reset_index()
)

repeated_ctrp_pairs = (
    ctrp_pair_summary.loc[
        ctrp_pair_summary["experiments"] > 1
    ]
    .copy()
)

repeated_ctrp_pairs["auc_range"] = (
    repeated_ctrp_pairs["auc_max"]
    - repeated_ctrp_pairs["auc_min"]
)

print(
    "Phase 6 CTRP model-compound pairs:",
    len(ctrp_pair_summary),
)
print(
    "Pairs with repeated experiments:",
    len(repeated_ctrp_pairs),
)
print(
    "Repeated-pair percentage:",
    f"{100 * len(repeated_ctrp_pairs) / len(ctrp_pair_summary):.2f}%",
)
print(
    "Maximum experiments per pair:",
    int(ctrp_pair_summary["experiments"].max()),
)

print("\nAUC range among repeated pairs:")
print(
    repeated_ctrp_pairs["auc_range"]
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99, 1.00])
)

Phase 6 CTRP model-compound pairs: 357460
Pairs with repeated experiments: 7708
Repeated-pair percentage: 2.16%
Maximum experiments per pair: 3

AUC range among repeated pairs:
0.50     0.765500
0.75     1.521500
0.90     2.550270
0.95     3.393020
0.99     5.698351
1.00    11.500000
Name: auc_range, dtype: float64


In [35]:
# =============================================================================
# Construct primary aggregated CTRP Phase 6 response
# =============================================================================

ctrp_primary_response = (
    ctrp_auc_phase6
    .groupby(
        ["ModelID", "master_cpd_id"],
        as_index=False,
    )
    .agg(
        response_value=(
            "area_under_curve",
            "median",
        ),
        n_experiments=(
            "experiment_id",
            "nunique",
        ),
    )
)

ctrp_primary_response["response_metric"] = (
    "area_under_curve"
)

ctrp_primary_response["aggregation_rule"] = (
    "median_across_experiments"
)

print(
    "Primary CTRP model-compound observations:",
    len(ctrp_primary_response),
)
print(
    "Unique ModelID-compound pairs:",
    ctrp_primary_response[
        ["ModelID", "master_cpd_id"]
    ]
    .drop_duplicates()
    .shape[0],
)
print(
    "Pairs based on one experiment:",
    (ctrp_primary_response["n_experiments"] == 1).sum(),
)
print(
    "Pairs based on repeated experiments:",
    (ctrp_primary_response["n_experiments"] > 1).sum(),
)
print(
    "Missing primary response values:",
    ctrp_primary_response["response_value"]
    .isna()
    .sum(),
)

Primary CTRP model-compound observations: 357460
Unique ModelID-compound pairs: 357460
Pairs based on one experiment: 349752
Pairs based on repeated experiments: 7708
Missing primary response values: 0


In [36]:
# =============================================================================
# Evaluate candidate drug estimability and lineage-support rule
# =============================================================================

MIN_MODELS_PER_LINEAGE = 20
MIN_SUPPORTED_LINEAGES = 3
MIN_SUPPORTED_MODELS = 100


def summarize_drug_eligibility(
    drug_model_pairs,
    drug_id_column,
    resource,
):
    pairs_with_lineage = (
        drug_model_pairs[
            [drug_id_column, "ModelID"]
        ]
        .drop_duplicates()
        .merge(
            phase6_score_universe[
                ["ModelID", "OncotreeLineage"]
            ],
            on="ModelID",
            how="inner",
            validate="many_to_one",
        )
    )

    lineage_counts = (
        pairs_with_lineage
        .groupby(
            [drug_id_column, "OncotreeLineage"]
        )["ModelID"]
        .nunique()
        .rename("models")
        .reset_index()
    )

    supported_lineages = (
        lineage_counts.loc[
            lineage_counts["models"]
            >= MIN_MODELS_PER_LINEAGE
        ]
        .copy()
    )

    eligibility = (
        supported_lineages
        .groupby(drug_id_column)
        .agg(
            supported_lineages=(
                "OncotreeLineage",
                "nunique",
            ),
            supported_models=(
                "models",
                "sum",
            ),
        )
        .reindex(
            drug_model_pairs[
                drug_id_column
            ].drop_duplicates()
        )
        .fillna(0)
        .reset_index()
    )

    eligibility[
        ["supported_lineages", "supported_models"]
    ] = eligibility[
        ["supported_lineages", "supported_models"]
    ].astype(int)

    eligibility["eligible"] = (
        (
            eligibility["supported_lineages"]
            >= MIN_SUPPORTED_LINEAGES
        )
        & (
            eligibility["supported_models"]
            >= MIN_SUPPORTED_MODELS
        )
    )

    eligibility["resource"] = resource

    return eligibility


gdsc_eligibility = summarize_drug_eligibility(
    gdsc_drug_models[
        ["DRUG_ID", "ModelID"]
    ],
    drug_id_column="DRUG_ID",
    resource="GDSC",
)

ctrp_eligibility = summarize_drug_eligibility(
    ctrp_primary_response[
        ["master_cpd_id", "ModelID"]
    ],
    drug_id_column="master_cpd_id",
    resource="CTRP",
)

prism_primary_assignment = (
    prism_cross_resource_compounds.loc[
        prism_cross_resource_compounds[
            "primary_prism_screen"
        ].notna()
        & (
            prism_cross_resource_compounds[
                "primary_prism_screen"
            ]
            != "AMBIGUOUS_NONREDO"
        ),
        [
            "prism_broad_id",
            "primary_prism_screen",
        ],
    ]
    .drop_duplicates()
)

prism_primary_pairs = (
    prism_drug_models
    .rename(
        columns={
            "depmap_id": "ModelID",
        }
    )
    .merge(
        prism_primary_assignment,
        left_on="broad_id",
        right_on="prism_broad_id",
        how="inner",
        validate="many_to_one",
    )
)

prism_primary_pairs = (
    prism_primary_pairs.loc[
        prism_primary_pairs["screen_id"]
        == prism_primary_pairs[
            "primary_prism_screen"
        ],
        ["broad_id", "ModelID"],
    ]
    .drop_duplicates()
)

prism_eligibility = summarize_drug_eligibility(
    prism_primary_pairs,
    drug_id_column="broad_id",
    resource="PRISM_exact_cross_resource",
)

eligibility_summary = pd.DataFrame(
    [
        {
            "resource": "GDSC",
            "drugs_evaluated": len(gdsc_eligibility),
            "drugs_eligible": gdsc_eligibility["eligible"].sum(),
        },
        {
            "resource": "CTRP",
            "drugs_evaluated": len(ctrp_eligibility),
            "drugs_eligible": ctrp_eligibility["eligible"].sum(),
        },
        {
            "resource": "PRISM_exact_cross_resource",
            "drugs_evaluated": len(prism_eligibility),
            "drugs_eligible": prism_eligibility["eligible"].sum(),
        },
    ]
)

eligibility_summary["eligible_fraction"] = (
    eligibility_summary["drugs_eligible"]
    / eligibility_summary["drugs_evaluated"]
)

display(eligibility_summary)

,resource,drugs_evaluated,drugs_eligible,eligible_fraction
0,GDSC,295,281,0.952542
1,CTRP,545,499,0.915596
2,PRISM_exact_cross_resource,197,194,0.984772


### Primary drug estimability rule

An outcome-blind coverage assessment was used to evaluate the prespecified
candidate eligibility rule before any program–drug association results were
inspected.

The primary Phase 6 rule is frozen as:

- at least 20 response-covered models per supported lineage;
- at least 3 supported lineages; and
- at least 100 models in total across supported lineages.

Only supported lineages contribute to the primary drug-specific model.

This rule retains 281/295 GDSC drugs, 499/545 CTRP compounds, and 194/197
PRISM exact cross-resource compounds with a unique primary screen assignment.

The authoritative methodological definition and rationale are recorded in
`docs/contracts/phase6/PHASE6_ANALYSIS_CONTRACT.md`.

In [37]:
# =============================================================================
# Construct primary PRISM Phase 6 response
# =============================================================================

prism_response = pd.read_csv(
    pharmacogenomic_input_paths[("prism", "response")],
    usecols=[
        "broad_id",
        "depmap_id",
        "screen_id",
        "auc",
    ],
)

prism_primary_response = (
    prism_response
    .rename(
        columns={
            "depmap_id": "ModelID",
            "auc": "response_value",
        }
    )
    .loc[
        lambda data: data["ModelID"].isin(
            phase6_scoreable_ids
        )
    ]
    .merge(
        prism_primary_assignment,
        left_on="broad_id",
        right_on="prism_broad_id",
        how="inner",
        validate="many_to_one",
    )
    .loc[
        lambda data:
        data["screen_id"]
        == data["primary_prism_screen"]
    ]
    .drop(
        columns="prism_broad_id"
    )
    .reset_index(drop=True)
)

prism_primary_response["response_metric"] = "auc"

prism_pair_duplicates = (
    prism_primary_response
    .duplicated(
        subset=["ModelID", "broad_id"],
        keep=False,
    )
)

print(
    "Primary PRISM response rows:",
    len(prism_primary_response),
)
print(
    "Unique ModelID-compound pairs:",
    prism_primary_response[
        ["ModelID", "broad_id"]
    ]
    .drop_duplicates()
    .shape[0],
)
print(
    "Duplicated ModelID-compound pairs:",
    prism_pair_duplicates.sum(),
)
print(
    "Missing primary response values:",
    prism_primary_response[
        "response_value"
    ]
    .isna()
    .sum(),
)

print("\nPrimary PRISM observations by selected screen:")
print(
    prism_primary_response["screen_id"]
    .value_counts()
    .sort_index()
)

Primary PRISM response rows: 84802
Unique ModelID-compound pairs: 84802
Duplicated ModelID-compound pairs: 0
Missing primary response values: 0

Primary PRISM observations by selected screen:
screen_id
HTS002    67078
MTS006     1818
MTS010    15906
Name: count, dtype: int64


In [38]:
# =============================================================================
# Construct frozen lineage-supported pharmacogenomic analysis universes
# =============================================================================

def restrict_to_supported_drug_lineages(
    response_data,
    eligibility_data,
    drug_id_column,
    resource,
):
    eligible_drugs = set(
        eligibility_data.loc[
            eligibility_data["eligible"],
            drug_id_column,
        ]
    )

    response_with_lineage = (
        response_data
        .loc[
            lambda data:
            data[drug_id_column].isin(
                eligible_drugs
            )
        ]
        .merge(
            phase6_score_universe[
                ["ModelID", "OncotreeLineage"]
            ],
            on="ModelID",
            how="inner",
            validate="many_to_one",
        )
    )

    supported_drug_lineages = (
        response_with_lineage
        .groupby(
            [drug_id_column, "OncotreeLineage"]
        )["ModelID"]
        .nunique()
        .rename("models")
        .reset_index()
        .loc[
            lambda data:
            data["models"]
            >= MIN_MODELS_PER_LINEAGE
        ]
    )

    primary_universe = (
        response_with_lineage
        .merge(
            supported_drug_lineages[
                [drug_id_column, "OncotreeLineage"]
            ],
            on=[
                drug_id_column,
                "OncotreeLineage",
            ],
            how="inner",
            validate="many_to_one",
        )
        .reset_index(drop=True)
    )

    summary = {
        "resource": resource,
        "eligible_drugs": (
            primary_universe[
                drug_id_column
            ].nunique()
        ),
        "primary_observations": len(
            primary_universe
        ),
        "unique_models": (
            primary_universe["ModelID"]
            .nunique()
        ),
        "represented_lineages": (
            primary_universe[
                "OncotreeLineage"
            ].nunique()
        ),
    }

    return (
        primary_universe,
        supported_drug_lineages,
        summary,
    )


gdsc_primary_universe, gdsc_supported_lineages, gdsc_primary_summary = (
    restrict_to_supported_drug_lineages(
        gdsc_drug_models[
            ["DRUG_ID", "ModelID"]
        ],
        gdsc_eligibility,
        drug_id_column="DRUG_ID",
        resource="GDSC",
    )
)

ctrp_primary_universe, ctrp_supported_lineages, ctrp_primary_summary = (
    restrict_to_supported_drug_lineages(
        ctrp_primary_response,
        ctrp_eligibility,
        drug_id_column="master_cpd_id",
        resource="CTRP",
    )
)

prism_primary_universe, prism_supported_lineages, prism_primary_summary = (
    restrict_to_supported_drug_lineages(
        prism_primary_response,
        prism_eligibility,
        drug_id_column="broad_id",
        resource="PRISM_exact_cross_resource",
    )
)

primary_universe_summary = pd.DataFrame(
    [
        gdsc_primary_summary,
        ctrp_primary_summary,
        prism_primary_summary,
    ]
)

display(primary_universe_summary)

,resource,eligible_drugs,primary_observations,unique_models,represented_lineages
0,GDSC,281,136176,557,11
1,CTRP,499,302310,736,15
2,PRISM_exact_cross_resource,194,65736,386,11


In [39]:
# =============================================================================
# Load minimal GDSC response columns required for the Phase 6 primary response
# =============================================================================

gdsc_response = pd.read_excel(
    pharmacogenomic_input_paths[("gdsc", "response")],
    usecols=[
        "SANGER_MODEL_ID",
        "DRUG_ID",
        "LN_IC50",
    ],
)

In [40]:
# =============================================================================
# Construct primary GDSC Phase 6 response from the frozen model crosswalk
# =============================================================================

gdsc_model_crosswalk = (
    phase6_anchor_cohort[
        ["SangerModelID", "ModelID"]
    ]
    .rename(
        columns={
            "SangerModelID": "SANGER_MODEL_ID",
        }
    )
)

gdsc_primary_response = (
    gdsc_response
    .merge(
        gdsc_model_crosswalk,
        on="SANGER_MODEL_ID",
        how="inner",
        validate="many_to_one",
    )
    .rename(
        columns={
            "LN_IC50": "response_value",
        }
    )
    [
        [
            "ModelID",
            "DRUG_ID",
            "response_value",
        ]
    ]
)

gdsc_primary_response["response_metric"] = "LN_IC50"

gdsc_primary_universe = (
    gdsc_primary_universe[
        [
            "DRUG_ID",
            "ModelID",
            "OncotreeLineage",
        ]
    ]
    .merge(
        gdsc_primary_response,
        on=["DRUG_ID", "ModelID"],
        how="left",
        validate="one_to_one",
    )
)

print(
    "Primary GDSC response pairs:",
    len(gdsc_primary_response),
)
print(
    "Duplicated ModelID-drug pairs:",
    gdsc_primary_response
    .duplicated(
        subset=["ModelID", "DRUG_ID"]
    )
    .sum(),
)
print(
    "Missing primary response values:",
    gdsc_primary_response[
        "response_value"
    ]
    .isna()
    .sum(),
)

print(
    "\nLineage-supported primary GDSC observations:",
    len(gdsc_primary_universe),
)
print(
    "Eligible drugs:",
    gdsc_primary_universe[
        "DRUG_ID"
    ]
    .nunique(),
)
print(
    "Missing response values after support restriction:",
    gdsc_primary_universe[
        "response_value"
    ]
    .isna()
    .sum(),
)

Primary GDSC response pairs: 180614
Duplicated ModelID-drug pairs: 0
Missing primary response values: 0

Lineage-supported primary GDSC observations: 136176
Eligible drugs: 281
Missing response values after support restriction: 0


In [41]:
# =============================================================================
# Assemble the primary GDSC program–drug analysis frame
# =============================================================================

PROGRAM_COLUMNS = [
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
]

gdsc_primary_analysis = (
    gdsc_primary_universe
    .merge(
        phase6_score_universe[
            ["ModelID", *PROGRAM_COLUMNS]
        ],
        on="ModelID",
        how="left",
        validate="many_to_one",
    )
)

print(
    "Primary GDSC analysis observations:",
    len(gdsc_primary_analysis),
)
print(
    "Eligible drugs:",
    gdsc_primary_analysis["DRUG_ID"].nunique(),
)
print(
    "Unique models:",
    gdsc_primary_analysis["ModelID"].nunique(),
)
print(
    "Supported lineages:",
    gdsc_primary_analysis["OncotreeLineage"].nunique(),
)

print("\nMissing values:")
print(
    gdsc_primary_analysis[
        [
            "response_value",
            "OncotreeLineage",
            *PROGRAM_COLUMNS,
        ]
    ]
    .isna()
    .sum()
)

Primary GDSC analysis observations: 136176
Eligible drugs: 281
Unique models: 557
Supported lineages: 11

Missing values:
response_value     0
OncotreeLineage    0
CONSENSUS_TX_01    0
CONSENSUS_TX_02    0
CONSENSUS_TX_03    0
dtype: int64


In [42]:
# =============================================================================
# Verify algebraic estimability of the candidate primary association model
# =============================================================================

design_checks = []

for drug_id, drug_data in gdsc_primary_analysis.groupby(
    "DRUG_ID",
    sort=False,
):
    lineage_dummies = pd.get_dummies(
        drug_data["OncotreeLineage"],
        drop_first=True,
        dtype=float,
    )

    base_design = np.column_stack(
        [
            np.ones(len(drug_data)),
            lineage_dummies.to_numpy(),
        ]
    )

    response_has_variation = (
        drug_data["response_value"].nunique()
        > 1
    )

    for program in PROGRAM_COLUMNS:
        program_values = (
            drug_data[program]
            .to_numpy(dtype=float)
        )

        design_matrix = np.column_stack(
            [
                base_design,
                program_values,
            ]
        )

        design_rank = np.linalg.matrix_rank(
            design_matrix
        )

        design_checks.append(
            {
                "DRUG_ID": drug_id,
                "program": program,
                "n_models": len(drug_data),
                "n_lineages": (
                    drug_data[
                        "OncotreeLineage"
                    ].nunique()
                ),
                "response_has_variation": (
                    response_has_variation
                ),
                "program_has_variation": (
                    np.ptp(program_values) > 0
                ),
                "design_columns": (
                    design_matrix.shape[1]
                ),
                "design_rank": design_rank,
                "full_rank": (
                    design_rank
                    == design_matrix.shape[1]
                ),
            }
        )

design_checks = pd.DataFrame(
    design_checks
)

print(
    "Candidate drug-program models:",
    len(design_checks),
)
print(
    "Full-rank design matrices:",
    design_checks["full_rank"].sum(),
)
print(
    "Non-full-rank design matrices:",
    (~design_checks["full_rank"]).sum(),
)
print(
    "Models with constant response:",
    (~design_checks["response_has_variation"]).sum(),
)
print(
    "Models with constant program score:",
    (~design_checks["program_has_variation"]).sum(),
)

display(
    design_checks.loc[
        (~design_checks["full_rank"])
        | (~design_checks["response_has_variation"])
        | (~design_checks["program_has_variation"])
    ]
)

Candidate drug-program models: 843
Full-rank design matrices: 843
Non-full-rank design matrices: 0
Models with constant response: 0
Models with constant program score: 0


,DRUG_ID,program,n_models,n_lineages,response_has_variation,program_has_variation,design_columns,design_rank,full_rank


### Frozen primary association specification

Before inspection of any program–drug association result, the notebook-600
primary inferential family was frozen as 281 eligible GDSC drugs × 3 frozen
consensus programs = 843 tests.

Each association is estimated as:

`LN_IC50 ~ program_score + C(OncotreeLineage)`

using OLS with HC3 robust standard errors and only drug-specific supported
lineages under the frozen `20 / 3 / 100` rule.

Raw GDSC `LN_IC50` and the frozen Phase 4 score scale are retained without
drug-specific re-standardization.

Benjamini-Hochberg correction is applied jointly across all 843 primary tests,
with `q < 0.05` defining the FDR-controlled developmental association set.

CTRP and PRISM are not used in this primary family and remain reserved for
cross-screen replication.

Leave-one-lineage-out analyses are prespecified sensitivity characterizations
and do not alter primary inference.

In [43]:
# =============================================================================
# Fit the frozen primary GDSC program–drug association family
# =============================================================================

from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests


primary_association_rows = []

for drug_id, drug_data in gdsc_primary_analysis.groupby(
    "DRUG_ID",
    sort=False,
):
    for program in PROGRAM_COLUMNS:
        model = ols(
            formula=(
                f"response_value ~ "
                f"{program} + C(OncotreeLineage)"
            ),
            data=drug_data,
        ).fit(
            cov_type="HC3",
            use_t=True,
        )

        confidence_interval = model.conf_int(
            alpha=0.05
        ).loc[program]

        primary_association_rows.append(
            {
                "resource": "GDSC",
                "evidence_role": "developmental_internal",
                "DRUG_ID": drug_id,
                "program": program,
                "response_metric": "LN_IC50",
                "n_models": len(drug_data),
                "n_lineages": (
                    drug_data[
                        "OncotreeLineage"
                    ].nunique()
                ),
                "beta": model.params[program],
                "se_hc3": model.bse[program],
                "ci95_low": confidence_interval.iloc[0],
                "ci95_high": confidence_interval.iloc[1],
                "t_value": model.tvalues[program],
                "p_value": model.pvalues[program],
                "df_resid": model.df_resid,
            }
        )

gdsc_primary_associations = pd.DataFrame(
    primary_association_rows
)

assert len(gdsc_primary_associations) == 843
assert gdsc_primary_associations["p_value"].notna().all()
assert np.isfinite(
    gdsc_primary_associations["p_value"]
).all()

reject_fdr, q_values, _, _ = multipletests(
    gdsc_primary_associations["p_value"],
    alpha=0.05,
    method="fdr_bh",
)

gdsc_primary_associations["q_value"] = q_values
gdsc_primary_associations["fdr_05"] = reject_fdr

print(
    "Primary tests fitted:",
    len(gdsc_primary_associations),
)
print(
    "Unique drugs:",
    gdsc_primary_associations[
        "DRUG_ID"
    ].nunique(),
)
print(
    "Programs:",
    gdsc_primary_associations[
        "program"
    ].nunique(),
)
print(
    "Missing p-values:",
    gdsc_primary_associations[
        "p_value"
    ].isna()
    .sum(),
)
print(
    "BH FDR q < 0.05:",
    gdsc_primary_associations[
        "fdr_05"
    ].sum(),
)

display(
    gdsc_primary_associations.head(12)
)

Primary tests fitted: 843
Unique drugs: 281
Programs: 3
Missing p-values: 0
BH FDR q < 0.05: 338


,resource,evidence_role,DRUG_ID,program,response_metric,n_models,n_lineages,beta,se_hc3,ci95_low,ci95_high,t_value,p_value,df_resid,q_value,fdr_05
0,GDSC,developmental_internal,1003,CONSENSUS_TX_01,LN_IC50,556,11,-0.752874,0.185047,-1.116368,-0.389380,-4.068553,5.431179e-05,544.0,0.000627,True
1,GDSC,developmental_internal,1003,CONSENSUS_TX_02,LN_IC50,556,11,-0.339577,0.106594,-0.548964,-0.130191,-3.185700,1.526829e-03,544.0,0.006739,True
2,GDSC,developmental_internal,1003,CONSENSUS_TX_03,LN_IC50,556,11,0.044303,0.120183,-0.191776,0.280383,0.368632,7.125453e-01,544.0,0.799835,False
3,GDSC,developmental_internal,1004,CONSENSUS_TX_01,LN_IC50,419,10,-0.696002,0.217456,-1.123475,-0.268528,-3.200660,1.478386e-03,408.0,0.006688,True
4,GDSC,developmental_internal,1004,CONSENSUS_TX_02,LN_IC50,419,10,-0.872190,0.173276,-1.212815,-0.531565,-5.033536,7.236997e-07,408.0,0.000031,True
5,GDSC,developmental_internal,1004,CONSENSUS_TX_03,LN_IC50,419,10,-0.221201,0.170431,-0.556234,0.113832,-1.297890,1.950585e-01,408.0,0.298429,False
6,GDSC,developmental_internal,1005,CONSENSUS_TX_01,LN_IC50,432,10,-0.624917,0.223461,-1.064155,-0.185678,-2.796533,5.402090e-03,421.0,0.019134,True
7,GDSC,developmental_internal,1005,CONSENSUS_TX_02,LN_IC50,432,10,-0.457965,0.119118,-0.692106,-0.223824,-3.844619,1.393923e-04,421.0,0.001187,True
8,GDSC,developmental_internal,1005,CONSENSUS_TX_03,LN_IC50,432,10,0.036524,0.122668,-0.204593,0.277642,0.297748,7.660423e-01,421.0,0.838667,False
9,GDSC,developmental_internal,1006,CONSENSUS_TX_01,LN_IC50,420,10,-0.676444,0.300773,-1.267698,-0.085191,-2.249022,2.504253e-02,409.0,0.058833,False


In [44]:
# =============================================================================
# Run frozen leave-one-supported-lineage-out association refits
# =============================================================================

primary_beta_lookup = (
    gdsc_primary_associations
    .set_index(["DRUG_ID", "program"])["beta"]
    .to_dict()
)

lolo_rows = []

for drug_id, drug_data in gdsc_primary_analysis.groupby(
    "DRUG_ID",
    sort=False,
):
    supported_lineages = (
        drug_data["OncotreeLineage"]
        .drop_duplicates()
        .tolist()
    )

    for program in PROGRAM_COLUMNS:
        primary_beta = primary_beta_lookup[
            (drug_id, program)
        ]

        for omitted_lineage in supported_lineages:
            refit_data = drug_data.loc[
                drug_data["OncotreeLineage"]
                != omitted_lineage
            ]

            refit = ols(
                formula=(
                    f"response_value ~ "
                    f"{program} + C(OncotreeLineage)"
                ),
                data=refit_data,
            ).fit(
                cov_type="HC3",
                use_t=True,
            )

            lolo_beta = refit.params[program]

            lolo_rows.append(
                {
                    "DRUG_ID": drug_id,
                    "program": program,
                    "omitted_lineage": omitted_lineage,
                    "n_models_refit": len(refit_data),
                    "n_lineages_refit": (
                        refit_data[
                            "OncotreeLineage"
                        ].nunique()
                    ),
                    "primary_beta": primary_beta,
                    "lolo_beta": lolo_beta,
                    "beta_change": (
                        lolo_beta - primary_beta
                    ),
                    "abs_beta_change": abs(
                        lolo_beta - primary_beta
                    ),
                    "same_sign_as_primary": (
                        np.sign(lolo_beta)
                        == np.sign(primary_beta)
                    ),
                }
            )

gdsc_lolo_refits = pd.DataFrame(
    lolo_rows
)

expected_lolo_refits = int(
    gdsc_primary_associations[
        "n_lineages"
    ].sum()
)

assert len(gdsc_lolo_refits) == expected_lolo_refits
assert gdsc_lolo_refits["lolo_beta"].notna().all()
assert np.isfinite(
    gdsc_lolo_refits["lolo_beta"]
).all()

print(
    "LOLO refits completed:",
    len(gdsc_lolo_refits),
)
print(
    "Expected LOLO refits:",
    expected_lolo_refits,
)
print(
    "Drug-program associations covered:",
    gdsc_lolo_refits[
        ["DRUG_ID", "program"]
    ]
    .drop_duplicates()
    .shape[0],
)
print(
    "Missing LOLO coefficients:",
    gdsc_lolo_refits[
        "lolo_beta"
    ]
    .isna()
    .sum(),
)

LOLO refits completed: 8883
Expected LOLO refits: 8883
Drug-program associations covered: 843
Missing LOLO coefficients: 0


In [45]:
# =============================================================================
# Summarize frozen leave-one-lineage-out sensitivity metrics
# =============================================================================

gdsc_lolo_refits["same_sign_as_primary"] = (
    gdsc_lolo_refits["lolo_beta"]
    * gdsc_lolo_refits["primary_beta"]
    > 0
)

gdsc_lolo_refits["sign_reversal"] = (
    gdsc_lolo_refits["lolo_beta"]
    * gdsc_lolo_refits["primary_beta"]
    < 0
)

lolo_summary_base = (
    gdsc_lolo_refits
    .groupby(
        ["DRUG_ID", "program"],
        sort=False,
    )
    .agg(
        n_lolo_refits=(
            "omitted_lineage",
            "size",
        ),
        lolo_beta_min=(
            "lolo_beta",
            "min",
        ),
        lolo_beta_max=(
            "lolo_beta",
            "max",
        ),
        lolo_beta_median=(
            "lolo_beta",
            "median",
        ),
        lolo_same_sign_fraction=(
            "same_sign_as_primary",
            "mean",
        ),
        lolo_any_sign_reversal=(
            "sign_reversal",
            "any",
        ),
        lolo_max_abs_beta_change=(
            "abs_beta_change",
            "max",
        ),
    )
    .reset_index()
)

max_change_index = (
    gdsc_lolo_refits
    .groupby(
        ["DRUG_ID", "program"],
        sort=False,
    )["abs_beta_change"]
    .idxmax()
)

lolo_max_change_lineage = (
    gdsc_lolo_refits.loc[
        max_change_index,
        [
            "DRUG_ID",
            "program",
            "omitted_lineage",
        ],
    ]
    .rename(
        columns={
            "omitted_lineage":
                "lolo_max_change_omitted_lineage",
        }
    )
)

gdsc_lolo_summary = (
    lolo_summary_base
    .merge(
        lolo_max_change_lineage,
        on=["DRUG_ID", "program"],
        how="left",
        validate="one_to_one",
    )
)

assert len(gdsc_lolo_summary) == 843
assert gdsc_lolo_summary[
    "lolo_max_change_omitted_lineage"
].notna().all()

print(
    "LOLO summaries:",
    len(gdsc_lolo_summary),
)
print(
    "Associations with any sign reversal:",
    gdsc_lolo_summary[
        "lolo_any_sign_reversal"
    ].sum(),
)
print(
    "Minimum sign-concordance fraction:",
    gdsc_lolo_summary[
        "lolo_same_sign_fraction"
    ].min(),
)
print(
    "Median sign-concordance fraction:",
    gdsc_lolo_summary[
        "lolo_same_sign_fraction"
    ].median(),
)

display(
    gdsc_lolo_summary.head(12)
)

LOLO summaries: 843
Associations with any sign reversal: 175
Minimum sign-concordance fraction: 0.2727272727272727
Median sign-concordance fraction: 1.0


,DRUG_ID,program,n_lolo_refits,lolo_beta_min,lolo_beta_max,lolo_beta_median,lolo_same_sign_fraction,lolo_any_sign_reversal,lolo_max_abs_beta_change,lolo_max_change_omitted_lineage
0,1003,CONSENSUS_TX_01,11,-1.656627,-0.571997,-0.721819,1.000000,False,0.903753,Lymphoid
1,1003,CONSENSUS_TX_02,11,-0.406240,-0.222935,-0.339710,1.000000,False,0.116642,Lung
2,1003,CONSENSUS_TX_03,11,-0.051540,0.123745,0.043165,0.818182,True,0.095843,CNS/Brain
3,1004,CONSENSUS_TX_01,10,-1.927327,-0.382162,-0.674170,1.000000,False,1.231325,Lymphoid
4,1004,CONSENSUS_TX_02,10,-0.983738,-0.797469,-0.883747,1.000000,False,0.111548,Esophagus/Stomach
5,1004,CONSENSUS_TX_03,10,-0.336188,-0.139977,-0.220105,1.000000,False,0.114987,CNS/Brain
6,1005,CONSENSUS_TX_01,10,-1.781096,-0.315811,-0.601643,1.000000,False,1.156179,Lymphoid
7,1005,CONSENSUS_TX_02,10,-0.563651,-0.311683,-0.460982,1.000000,False,0.146282,Lung
8,1005,CONSENSUS_TX_03,10,-0.056743,0.095827,0.046257,0.700000,True,0.093267,CNS/Brain
9,1006,CONSENSUS_TX_01,10,-2.005237,-0.527269,-0.638726,1.000000,False,1.328793,Lymphoid


In [46]:
# =============================================================================
# Integrate primary GDSC associations with frozen LOLO sensitivity summaries
# =============================================================================

gdsc_primary_associations_with_lolo = (
    gdsc_primary_associations
    .merge(
        gdsc_lolo_summary,
        on=["DRUG_ID", "program"],
        how="left",
        validate="one_to_one",
    )
)

assert len(gdsc_primary_associations_with_lolo) == 843
assert gdsc_primary_associations_with_lolo[
    "n_lolo_refits"
].notna().all()

lolo_by_primary_status = (
    gdsc_primary_associations_with_lolo
    .groupby(
        "fdr_05",
        observed=True,
    )
    .agg(
        associations=("DRUG_ID", "size"),
        any_sign_reversal=(
            "lolo_any_sign_reversal",
            "sum",
        ),
        median_sign_concordance=(
            "lolo_same_sign_fraction",
            "median",
        ),
        minimum_sign_concordance=(
            "lolo_same_sign_fraction",
            "min",
        ),
        median_max_abs_beta_change=(
            "lolo_max_abs_beta_change",
            "median",
        ),
        maximum_abs_beta_change=(
            "lolo_max_abs_beta_change",
            "max",
        ),
    )
    .reset_index()
)

lolo_by_primary_status[
    "sign_reversal_fraction"
] = (
    lolo_by_primary_status[
        "any_sign_reversal"
    ]
    / lolo_by_primary_status[
        "associations"
    ]
)

display(lolo_by_primary_status)

,fdr_05,associations,any_sign_reversal,median_sign_concordance,minimum_sign_concordance,median_max_abs_beta_change,maximum_abs_beta_change,sign_reversal_fraction
0,False,505,171,1.0,0.272727,0.096330,2.811486,0.338614
1,True,338,4,1.0,0.909091,0.118796,1.566984,0.011834


In [47]:
# =============================================================================
# Characterize the FDR-controlled GDSC association set by program and direction
# =============================================================================

gdsc_fdr_associations = (
    gdsc_primary_associations_with_lolo
    .loc[
        gdsc_primary_associations_with_lolo[
            "fdr_05"
        ]
    ]
    .copy()
)

gdsc_fdr_associations[
    "association_direction"
] = np.where(
    gdsc_fdr_associations["beta"] > 0,
    "resistance_like",
    "sensitivity_like",
)

fdr_program_summary = (
    gdsc_fdr_associations
    .groupby(
        ["program", "association_direction"],
        observed=True,
    )
    .agg(
        associations=("DRUG_ID", "size"),
        unique_drugs=("DRUG_ID", "nunique"),
        sign_reversal_cases=(
            "lolo_any_sign_reversal",
            "sum",
        ),
        median_sign_concordance=(
            "lolo_same_sign_fraction",
            "median",
        ),
        minimum_sign_concordance=(
            "lolo_same_sign_fraction",
            "min",
        ),
        median_beta=(
            "beta",
            "median",
        ),
    )
    .reset_index()
)

print(
    "FDR-controlled associations:",
    len(gdsc_fdr_associations),
)
print(
    "Unique drugs represented:",
    gdsc_fdr_associations[
        "DRUG_ID"
    ].nunique(),
)

display(fdr_program_summary)

print(
    "\nFDR associations with any LOLO sign reversal:"
)

display(
    gdsc_fdr_associations.loc[
        gdsc_fdr_associations[
            "lolo_any_sign_reversal"
        ],
        [
            "DRUG_ID",
            "program",
            "beta",
            "q_value",
            "n_models",
            "n_lineages",
            "lolo_same_sign_fraction",
            "lolo_beta_min",
            "lolo_beta_max",
            "lolo_max_abs_beta_change",
            "lolo_max_change_omitted_lineage",
        ],
    ]
)

FDR-controlled associations: 338
Unique drugs represented: 222


,program,association_direction,associations,unique_drugs,sign_reversal_cases,median_sign_concordance,minimum_sign_concordance,median_beta
0,CONSENSUS_TX_01,resistance_like,2,2,0,1.0,1.000000,0.409388
1,CONSENSUS_TX_01,sensitivity_like,119,119,0,1.0,1.000000,-0.464515
2,CONSENSUS_TX_02,resistance_like,20,20,3,1.0,0.909091,0.429117
3,CONSENSUS_TX_02,sensitivity_like,159,159,1,1.0,0.909091,-0.279157
4,CONSENSUS_TX_03,resistance_like,23,23,0,1.0,1.000000,0.288243
5,CONSENSUS_TX_03,sensitivity_like,15,15,0,1.0,1.000000,-0.275139



FDR associations with any LOLO sign reversal:


,DRUG_ID,program,beta,q_value,n_models,n_lineages,lolo_same_sign_fraction,lolo_beta_min,lolo_beta_max,lolo_max_abs_beta_change,lolo_max_change_omitted_lineage
31,1014,CONSENSUS_TX_02,0.308469,0.018090,541,11,0.909091,-0.042318,0.381706,0.350787,Lung
130,1060,CONSENSUS_TX_02,0.362755,0.017483,556,11,0.909091,-0.040428,0.482865,0.403183,Lung
241,1372,CONSENSUS_TX_02,0.432609,0.006541,556,11,0.909091,-0.040530,0.546483,0.473139,Lung
625,1910,CONSENSUS_TX_02,-0.314621,0.011954,549,11,0.909091,-0.382959,0.052921,0.367542,Lung


In [48]:
# =============================================================================
# Characterize cross-program overlap within the FDR-controlled GDSC drug set
# =============================================================================

eligible_gdsc_drugs = (
    gdsc_primary_associations[
        ["DRUG_ID"]
    ]
    .drop_duplicates()
    .copy()
)

fdr_by_drug = (
    gdsc_fdr_associations
    .groupby(
        "DRUG_ID",
        sort=False,
    )
    .agg(
        n_fdr_programs=(
            "program",
            "nunique",
        ),
        fdr_program_set=(
            "program",
            lambda values: "|".join(
                program
                for program in PROGRAM_COLUMNS
                if program in set(values)
            ),
        ),
        n_resistance_like=(
            "association_direction",
            lambda values: (
                values == "resistance_like"
            ).sum(),
        ),
        n_sensitivity_like=(
            "association_direction",
            lambda values: (
                values == "sensitivity_like"
            ).sum(),
        ),
    )
    .reset_index()
)

gdsc_fdr_drug_summary = (
    eligible_gdsc_drugs
    .merge(
        fdr_by_drug,
        on="DRUG_ID",
        how="left",
        validate="one_to_one",
    )
)

gdsc_fdr_drug_summary[
    "n_fdr_programs"
] = (
    gdsc_fdr_drug_summary[
        "n_fdr_programs"
    ]
    .fillna(0)
    .astype(int)
)

gdsc_fdr_drug_summary[
    "n_resistance_like"
] = (
    gdsc_fdr_drug_summary[
        "n_resistance_like"
    ]
    .fillna(0)
    .astype(int)
)

gdsc_fdr_drug_summary[
    "n_sensitivity_like"
] = (
    gdsc_fdr_drug_summary[
        "n_sensitivity_like"
    ]
    .fillna(0)
    .astype(int)
)

gdsc_fdr_drug_summary[
    "fdr_program_set"
] = (
    gdsc_fdr_drug_summary[
        "fdr_program_set"
    ]
    .fillna("none")
)

gdsc_fdr_drug_summary[
    "mixed_direction"
] = (
    (
        gdsc_fdr_drug_summary[
            "n_resistance_like"
        ] > 0
    )
    & (
        gdsc_fdr_drug_summary[
            "n_sensitivity_like"
        ] > 0
    )
)

program_count_summary = (
    gdsc_fdr_drug_summary[
        "n_fdr_programs"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("n_fdr_programs")
    .reset_index(name="drugs")
)

program_set_summary = (
    gdsc_fdr_drug_summary[
        "fdr_program_set"
    ]
    .value_counts()
    .rename_axis("fdr_program_set")
    .reset_index(name="drugs")
)

print(
    "Eligible GDSC drugs:",
    len(gdsc_fdr_drug_summary),
)
print(
    "Drugs with >=1 FDR association:",
    (
        gdsc_fdr_drug_summary[
            "n_fdr_programs"
        ] > 0
    ).sum(),
)
print(
    "Drugs with >=2 FDR programs:",
    (
        gdsc_fdr_drug_summary[
            "n_fdr_programs"
        ] >= 2
    ).sum(),
)
print(
    "Drugs with all 3 programs FDR:",
    (
        gdsc_fdr_drug_summary[
            "n_fdr_programs"
        ] == 3
    ).sum(),
)
print(
    "Multi-program drugs with mixed directions:",
    (
        (
            gdsc_fdr_drug_summary[
                "n_fdr_programs"
            ] >= 2
        )
        & gdsc_fdr_drug_summary[
            "mixed_direction"
        ]
    ).sum(),
)

display(program_count_summary)
display(program_set_summary)

Eligible GDSC drugs: 281
Drugs with >=1 FDR association: 222
Drugs with >=2 FDR programs: 110
Drugs with all 3 programs FDR: 6
Multi-program drugs with mixed directions: 12


,n_fdr_programs,drugs
0,0,59
1,1,112
2,2,104
3,3,6


,fdr_program_set,drugs
0,CONSENSUS_TX_01|CONSENSUS_TX_02,84
1,CONSENSUS_TX_02,72
2,none,59
3,CONSENSUS_TX_01,28
4,CONSENSUS_TX_02|CONSENSUS_TX_03,17
5,CONSENSUS_TX_03,12
6,CONSENSUS_TX_01|CONSENSUS_TX_02|CONSENSUS_TX_03,6
7,CONSENSUS_TX_01|CONSENSUS_TX_03,3


In [49]:
# =============================================================================
# Build GDSC pharmacological annotation for Phase 6 contextualization
# =============================================================================

gdsc_pharmacology = (
    pd.read_excel(
        pharmacogenomic_input_paths[
            ("gdsc", "response")
        ],
        usecols=[
            "DRUG_ID",
            "DRUG_NAME",
            "PUTATIVE_TARGET",
            "PATHWAY_NAME",
        ],
    )
    .drop_duplicates()
)

annotation_counts = (
    gdsc_pharmacology
    .groupby("DRUG_ID")
    .size()
)

eligible_drug_ids = set(
    gdsc_primary_associations[
        "DRUG_ID"
    ]
)

eligible_annotations = (
    gdsc_pharmacology.loc[
        gdsc_pharmacology[
            "DRUG_ID"
        ].isin(eligible_drug_ids)
    ]
)

print(
    "Distinct GDSC annotation rows:",
    len(gdsc_pharmacology),
)
print(
    "Unique annotated DRUG_IDs:",
    gdsc_pharmacology[
        "DRUG_ID"
    ].nunique(),
)
print(
    "DRUG_IDs with >1 annotation tuple:",
    (annotation_counts > 1).sum(),
)
print(
    "Eligible drugs represented:",
    eligible_annotations[
        "DRUG_ID"
    ].nunique(),
)
print(
    "Eligible drugs missing PUTATIVE_TARGET:",
    eligible_annotations[
        "PUTATIVE_TARGET"
    ].isna()
    .sum(),
)
print(
    "Eligible drugs missing PATHWAY_NAME:",
    eligible_annotations[
        "PATHWAY_NAME"
    ].isna()
    .sum(),
)

display(
    gdsc_pharmacology.loc[
        gdsc_pharmacology[
            "DRUG_ID"
        ].isin(
            annotation_counts.loc[
                annotation_counts > 1
            ].index
        )
    ]
    .sort_values("DRUG_ID")
)

Distinct GDSC annotation rows: 295
Unique annotated DRUG_IDs: 295
DRUG_IDs with >1 annotation tuple: 0
Eligible drugs represented: 281
Eligible drugs missing PUTATIVE_TARGET: 38
Eligible drugs missing PATHWAY_NAME: 0


,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,PATHWAY_NAME


In [50]:
# =============================================================================
# Attach GDSC pharmacological annotations to the primary association results
# =============================================================================

gdsc_primary_annotated = (
    gdsc_primary_associations_with_lolo
    .merge(
        gdsc_pharmacology[
            [
                "DRUG_ID",
                "DRUG_NAME",
                "PUTATIVE_TARGET",
                "PATHWAY_NAME",
            ]
        ],
        on="DRUG_ID",
        how="left",
        validate="many_to_one",
    )
)

assert len(gdsc_primary_annotated) == 843
assert gdsc_primary_annotated["DRUG_NAME"].notna().all()
assert gdsc_primary_annotated["PATHWAY_NAME"].notna().all()

gdsc_fdr_annotated = (
    gdsc_primary_annotated
    .loc[
        gdsc_primary_annotated["fdr_05"]
    ]
    .copy()
)

gdsc_fdr_annotated[
    "association_direction"
] = np.where(
    gdsc_fdr_annotated["beta"] > 0,
    "resistance_like",
    "sensitivity_like",
)

print(
    "Primary annotated associations:",
    len(gdsc_primary_annotated),
)
print(
    "FDR-controlled annotated associations:",
    len(gdsc_fdr_annotated),
)
print(
    "Unique FDR-associated drugs:",
    gdsc_fdr_annotated["DRUG_ID"].nunique(),
)
print(
    "Unique PATHWAY_NAME categories in FDR set:",
    gdsc_fdr_annotated["PATHWAY_NAME"].nunique(),
)
print(
    "FDR-associated drugs missing PUTATIVE_TARGET:",
    gdsc_fdr_annotated.loc[
        gdsc_fdr_annotated["PUTATIVE_TARGET"].isna(),
        "DRUG_ID",
    ].nunique(),
)

Primary annotated associations: 843
FDR-controlled annotated associations: 338
Unique FDR-associated drugs: 222
Unique PATHWAY_NAME categories in FDR set: 24
FDR-associated drugs missing PUTATIVE_TARGET: 32


In [51]:
# =============================================================================
# Characterize pathway structure across eligible and FDR-associated GDSC drugs
# =============================================================================

gdsc_eligible_drug_context = (
    gdsc_primary_annotated[
        [
            "DRUG_ID",
            "DRUG_NAME",
            "PUTATIVE_TARGET",
            "PATHWAY_NAME",
        ]
    ]
    .drop_duplicates()
    .merge(
        gdsc_fdr_drug_summary[
            [
                "DRUG_ID",
                "n_fdr_programs",
                "n_resistance_like",
                "n_sensitivity_like",
                "mixed_direction",
            ]
        ],
        on="DRUG_ID",
        how="left",
        validate="one_to_one",
    )
)

assert len(gdsc_eligible_drug_context) == 281

gdsc_eligible_drug_context[
    "has_fdr_association"
] = (
    gdsc_eligible_drug_context[
        "n_fdr_programs"
    ] > 0
)

pathway_drug_summary = (
    gdsc_eligible_drug_context
    .groupby(
        "PATHWAY_NAME",
        sort=True,
    )
    .agg(
        eligible_drugs=(
            "DRUG_ID",
            "size",
        ),
        fdr_associated_drugs=(
            "has_fdr_association",
            "sum",
        ),
        total_fdr_associations=(
            "n_fdr_programs",
            "sum",
        ),
        multi_program_fdr_drugs=(
            "n_fdr_programs",
            lambda values: (
                values >= 2
            ).sum(),
        ),
        mixed_direction_drugs=(
            "mixed_direction",
            "sum",
        ),
    )
    .reset_index()
)

pathway_drug_summary[
    "fdr_drug_fraction"
] = (
    pathway_drug_summary[
        "fdr_associated_drugs"
    ]
    / pathway_drug_summary[
        "eligible_drugs"
    ]
)

print(
    "Eligible pathway categories:",
    len(pathway_drug_summary),
)
print(
    "Pathways with >=1 FDR-associated drug:",
    (
        pathway_drug_summary[
            "fdr_associated_drugs"
        ] > 0
    ).sum(),
)
print(
    "Eligible drugs accounted for:",
    pathway_drug_summary[
        "eligible_drugs"
    ].sum(),
)
print(
    "FDR-associated drugs accounted for:",
    pathway_drug_summary[
        "fdr_associated_drugs"
    ].sum(),
)

display(pathway_drug_summary)

Eligible pathway categories: 24
Pathways with >=1 FDR-associated drug: 24
Eligible drugs accounted for: 281
FDR-associated drugs accounted for: 222


,PATHWAY_NAME,eligible_drugs,fdr_associated_drugs,total_fdr_associations,multi_program_fdr_drugs,mixed_direction_drugs,fdr_drug_fraction
0,ABL signaling,1,1,1,0,0,1.000000
1,Apoptosis regulation,13,10,17,7,0,0.769231
2,Cell cycle,13,10,15,5,0,0.769231
3,Chromatin histone acetylation,10,9,15,5,1,0.900000
4,Chromatin histone methylation,13,11,15,4,0,0.846154
5,Chromatin other,10,8,12,4,0,0.800000
6,Cytoskeleton,4,4,5,1,1,1.000000
7,DNA replication,21,20,34,13,1,0.952381
8,EGFR signaling,7,7,13,6,0,1.000000
9,ERK MAPK signaling,15,7,9,2,0,0.466667


In [52]:
# =============================================================================
# Characterize pathway context by frozen program and association direction
# =============================================================================

pathway_program_summary = (
    gdsc_fdr_annotated
    .groupby(
        [
            "PATHWAY_NAME",
            "program",
            "association_direction",
        ],
        observed=True,
        sort=True,
    )
    .agg(
        drugs=("DRUG_ID", "nunique"),
    )
    .reset_index()
)

pathway_program_totals = (
    gdsc_fdr_annotated
    .groupby(
        [
            "PATHWAY_NAME",
            "program",
        ],
        observed=True,
        sort=True,
    )
    .agg(
        fdr_drugs=("DRUG_ID", "nunique"),
    )
    .reset_index()
)

pathway_program_direction = (
    pathway_program_summary
    .pivot_table(
        index=[
            "PATHWAY_NAME",
            "program",
        ],
        columns="association_direction",
        values="drugs",
        fill_value=0,
    )
    .reset_index()
)

for direction in (
    "resistance_like",
    "sensitivity_like",
):
    if direction not in pathway_program_direction.columns:
        pathway_program_direction[
            direction
        ] = 0

pathway_program_direction = (
    pathway_program_totals
    .merge(
        pathway_program_direction,
        on=[
            "PATHWAY_NAME",
            "program",
        ],
        how="left",
        validate="one_to_one",
    )
)

eligible_pathway_counts = (
    gdsc_eligible_drug_context
    .groupby(
        "PATHWAY_NAME",
        sort=True,
    )["DRUG_ID"]
    .nunique()
    .rename("eligible_drugs")
    .reset_index()
)

pathway_program_direction = (
    pathway_program_direction
    .merge(
        eligible_pathway_counts,
        on="PATHWAY_NAME",
        how="left",
        validate="many_to_one",
    )
)

assert (
    pathway_program_direction[
        "resistance_like"
    ]
    + pathway_program_direction[
        "sensitivity_like"
    ]
    == pathway_program_direction[
        "fdr_drugs"
    ]
).all()

print(
    "Pathway-program combinations with >=1 FDR drug:",
    len(pathway_program_direction),
)
print(
    "Pathways represented:",
    pathway_program_direction[
        "PATHWAY_NAME"
    ].nunique(),
)
print(
    "Programs represented:",
    pathway_program_direction[
        "program"
    ].nunique(),
)

display(pathway_program_direction)

Pathway-program combinations with >=1 FDR drug: 58
Pathways represented: 24
Programs represented: 3


,PATHWAY_NAME,program,fdr_drugs,resistance_like,sensitivity_like,eligible_drugs
0,ABL signaling,CONSENSUS_TX_02,1,0.0,1.0,1
1,Apoptosis regulation,CONSENSUS_TX_01,7,0.0,7.0,13
2,Apoptosis regulation,CONSENSUS_TX_02,9,0.0,9.0,13
3,Apoptosis regulation,CONSENSUS_TX_03,1,0.0,1.0,13
4,Cell cycle,CONSENSUS_TX_01,8,0.0,8.0,13
5,Cell cycle,CONSENSUS_TX_02,7,0.0,7.0,13
6,Chromatin histone acetylation,CONSENSUS_TX_01,7,0.0,7.0,10
7,Chromatin histone acetylation,CONSENSUS_TX_02,7,0.0,7.0,10
8,Chromatin histone acetylation,CONSENSUS_TX_03,1,1.0,0.0,10
9,Chromatin histone methylation,CONSENSUS_TX_01,7,0.0,7.0,13


In [53]:
# =============================================================================
# Characterize GDSC PUTATIVE_TARGET annotation structure before target parsing
# =============================================================================

eligible_target_annotations = (
    gdsc_eligible_drug_context[
        [
            "DRUG_ID",
            "DRUG_NAME",
            "PUTATIVE_TARGET",
            "has_fdr_association",
        ]
    ]
    .copy()
)

nonmissing_targets = (
    eligible_target_annotations[
        "PUTATIVE_TARGET"
    ]
    .dropna()
    .astype(str)
)

fdr_nonmissing_targets = (
    eligible_target_annotations.loc[
        eligible_target_annotations[
            "has_fdr_association"
        ],
        "PUTATIVE_TARGET",
    ]
    .dropna()
    .astype(str)
)

target_delimiter_summary = pd.DataFrame(
    {
        "delimiter": [
            ",",
            ";",
            "/",
            "|",
        ],
        "eligible_strings": [
            nonmissing_targets.str.contains(
                delimiter,
                regex=False,
            ).sum()
            for delimiter in [
                ",",
                ";",
                "/",
                "|",
            ]
        ],
        "fdr_strings": [
            fdr_nonmissing_targets.str.contains(
                delimiter,
                regex=False,
            ).sum()
            for delimiter in [
                ",",
                ";",
                "/",
                "|",
            ]
        ],
    }
)

multi_target_examples = (
    eligible_target_annotations.loc[
        eligible_target_annotations[
            "PUTATIVE_TARGET"
        ]
        .fillna("")
        .str.contains(
            r"[,;/|]",
            regex=True,
        ),
        [
            "DRUG_ID",
            "DRUG_NAME",
            "PUTATIVE_TARGET",
            "has_fdr_association",
        ],
    ]
    .drop_duplicates()
    .sort_values(
        [
            "PUTATIVE_TARGET",
            "DRUG_ID",
        ]
    )
)

print(
    "Eligible drugs:",
    len(eligible_target_annotations),
)
print(
    "Eligible drugs with non-missing PUTATIVE_TARGET:",
    nonmissing_targets.shape[0],
)
print(
    "Unique non-missing target strings:",
    nonmissing_targets.nunique(),
)
print(
    "FDR-associated drugs with non-missing PUTATIVE_TARGET:",
    fdr_nonmissing_targets.shape[0],
)
print(
    "Unique FDR target strings:",
    fdr_nonmissing_targets.nunique(),
)

display(target_delimiter_summary)

display(
    multi_target_examples.head(40)
)

Eligible drugs: 281
Eligible drugs with non-missing PUTATIVE_TARGET: 243
Unique non-missing target strings: 174
FDR-associated drugs with non-missing PUTATIVE_TARGET: 190
Unique FDR target strings: 145


,delimiter,eligible_strings,fdr_strings
0,",",102,75
1,;,0,0
2,/,1,1
3,|,0,0


,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,has_fdr_association
51,1079,Dasatinib,"ABL, SRC, Ephrins, PDGFR, KIT",True
156,1783,LMB_AB1,"ADRA1A, ADRB1",True
218,1924,Ipatasertib,"AKT1, AKT, AKT3",False
38,1053,MK-2206,"AKT1, AKT2",True
96,1553,Uprosertib,"AKT1, AKT2, AKT3",True
152,1778,GSK2110183B,"AKT1, AKT2, AKT3",True
210,1912,Afuresertib,"AKT1, AKT2, AKT3",True
253,2106,Uprosertib,"AKT1, AKT2, AKT3",True
213,1916,AZD5363,"AKT1, AKT2, AKT3, ROCK2",False
35,1050,ZM447439,"AURKA, AURKB",True


In [54]:
# =============================================================================
# Inspect the single slash-containing GDSC target annotation
# =============================================================================

display(
    eligible_target_annotations.loc[
        eligible_target_annotations[
            "PUTATIVE_TARGET"
        ]
        .fillna("")
        .str.contains(
            "/",
            regex=False,
        ),
        [
            "DRUG_ID",
            "DRUG_NAME",
            "PUTATIVE_TARGET",
            "has_fdr_association",
        ],
    ]
)

,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,has_fdr_association
243,2040,Foretinib,"MET, KDR, TIE2, VEGFR3/FLT4, RON, PDGFR, FGFR1...",True


**Frozen descriptive target parsing rule.** `PUTATIVE_TARGET` is parsed only on commas, with surrounding whitespace removed. Null, empty, or whitespace-only annotations are treated as missing provider annotations. Other punctuation, including `/`, is retained as part of the provider-supplied target label. No synonym mapping, gene-symbol harmonization, manual curation, or target imputation is performed. Target annotations are used only for descriptive pharmacological contextualization and do not modify the primary GDSC association family or FDR status.

In [55]:
# =============================================================================
# Parse provider-supplied GDSC target annotations using the frozen rule
# =============================================================================

target_source = (
    gdsc_eligible_drug_context[
        [
            "DRUG_ID",
            "DRUG_NAME",
            "PATHWAY_NAME",
            "PUTATIVE_TARGET",
            "has_fdr_association",
            "n_fdr_programs",
        ]
    ]
    .copy()
)

target_source[
    "PUTATIVE_TARGET"
] = (
    target_source[
        "PUTATIVE_TARGET"
    ]
    .replace(r"^\s*$", np.nan, regex=True)
)

gdsc_target_long = (
    target_source.loc[
        target_source[
            "PUTATIVE_TARGET"
        ].notna()
    ]
    .assign(
        target_token=lambda data: (
            data["PUTATIVE_TARGET"]
            .str.split(",")
        )
    )
    .explode("target_token")
)

gdsc_target_long[
    "target_token"
] = (
    gdsc_target_long[
        "target_token"
    ]
    .str.strip()
)

assert gdsc_target_long[
    "target_token"
].ne("").all()

gdsc_target_long = (
    gdsc_target_long
    .drop_duplicates(
        [
            "DRUG_ID",
            "target_token",
        ]
    )
    .reset_index(drop=True)
)

target_counts_per_drug = (
    gdsc_target_long
    .groupby("DRUG_ID")[
        "target_token"
    ]
    .nunique()
)

print(
    "Eligible drugs with parsed targets:",
    gdsc_target_long[
        "DRUG_ID"
    ].nunique(),
)
print(
    "Eligible drugs without usable target annotation:",
    (
        target_source[
            "PUTATIVE_TARGET"
        ].isna()
    ).sum(),
)
print(
    "FDR-associated drugs with parsed targets:",
    gdsc_target_long.loc[
        gdsc_target_long[
            "has_fdr_association"
        ],
        "DRUG_ID",
    ].nunique(),
)
print(
    "FDR-associated drugs without usable target annotation:",
    (
        target_source[
            "has_fdr_association"
        ]
        & target_source[
            "PUTATIVE_TARGET"
        ].isna()
    ).sum(),
)
print(
    "Unique provider target tokens:",
    gdsc_target_long[
        "target_token"
    ].nunique(),
)
print(
    "Total drug-target pairs:",
    len(gdsc_target_long),
)
print(
    "Median targets per annotated drug:",
    target_counts_per_drug.median(),
)
print(
    "Maximum targets per annotated drug:",
    target_counts_per_drug.max(),
)

display(
    target_counts_per_drug
    .value_counts()
    .sort_index()
    .rename_axis("targets_per_drug")
    .reset_index(name="drugs")
)

Eligible drugs with parsed targets: 242
Eligible drugs without usable target annotation: 39
FDR-associated drugs with parsed targets: 189
FDR-associated drugs without usable target annotation: 33
Unique provider target tokens: 221
Total drug-target pairs: 417
Median targets per annotated drug: 1.0
Maximum targets per annotated drug: 8


,targets_per_drug,drugs
0,1,140
1,2,57
2,3,28
3,4,11
4,5,4
5,7,1
6,8,1


In [56]:
# =============================================================================
# Characterize recurrence of provider-supplied target tokens across eligible drugs
# =============================================================================

target_token_summary = (
    gdsc_target_long
    .groupby(
        "target_token",
        sort=True,
    )
    .agg(
        eligible_drugs=(
            "DRUG_ID",
            "nunique",
        ),
        fdr_associated_drugs=(
            "has_fdr_association",
            "sum",
        ),
        total_fdr_associations=(
            "n_fdr_programs",
            "sum",
        ),
    )
    .reset_index()
)

target_recurrence_summary = (
    target_token_summary[
        "eligible_drugs"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "eligible_drugs_per_target"
    )
    .reset_index(
        name="target_tokens"
    )
)

print(
    "Provider target tokens:",
    len(target_token_summary),
)
print(
    "Singleton target tokens:",
    (
        target_token_summary[
            "eligible_drugs"
        ] == 1
    ).sum(),
)
print(
    "Target tokens represented by >=2 eligible drugs:",
    (
        target_token_summary[
            "eligible_drugs"
        ] >= 2
    ).sum(),
)
print(
    "Target tokens represented by >=3 eligible drugs:",
    (
        target_token_summary[
            "eligible_drugs"
        ] >= 3
    ).sum(),
)
print(
    "Target tokens represented by >=5 eligible drugs:",
    (
        target_token_summary[
            "eligible_drugs"
        ] >= 5
    ).sum(),
)
print(
    "Target tokens with >=1 FDR-associated drug:",
    (
        target_token_summary[
            "fdr_associated_drugs"
        ] >= 1
    ).sum(),
)

display(target_recurrence_summary)

Provider target tokens: 221
Singleton target tokens: 132
Target tokens represented by >=2 eligible drugs: 89
Target tokens represented by >=3 eligible drugs: 48
Target tokens represented by >=5 eligible drugs: 19
Target tokens with >=1 FDR-associated drug: 193


,eligible_drugs_per_target,target_tokens
0,1,132
1,2,41
2,3,22
3,4,7
4,5,10
5,6,6
6,7,1
7,8,2


In [57]:
# =============================================================================
# Consolidate provider-supplied pharmacological context at the drug level
# =============================================================================

parsed_targets_by_drug = (
    gdsc_target_long
    .groupby(
        "DRUG_ID",
        sort=False,
    )
    .agg(
        parsed_target_count=(
            "target_token",
            "nunique",
        ),
        parsed_targets=(
            "target_token",
            lambda values: "|".join(
                sorted(set(values))
            ),
        ),
    )
    .reset_index()
)

gdsc_drug_pharmacology = (
    gdsc_eligible_drug_context
    .merge(
        parsed_targets_by_drug,
        on="DRUG_ID",
        how="left",
        validate="one_to_one",
    )
)

gdsc_drug_pharmacology[
    "parsed_target_count"
] = (
    gdsc_drug_pharmacology[
        "parsed_target_count"
    ]
    .fillna(0)
    .astype(int)
)

assert len(gdsc_drug_pharmacology) == 281
assert gdsc_drug_pharmacology["DRUG_ID"].nunique() == 281

print(
    "Eligible drugs:",
    len(gdsc_drug_pharmacology),
)
print(
    "FDR-associated drugs:",
    gdsc_drug_pharmacology[
        "has_fdr_association"
    ].sum(),
)
print(
    "Drugs with >=1 parsed target:",
    (
        gdsc_drug_pharmacology[
            "parsed_target_count"
        ] > 0
    ).sum(),
)
print(
    "Drugs without usable target annotation:",
    (
        gdsc_drug_pharmacology[
            "parsed_target_count"
        ] == 0
    ).sum(),
)
print(
    "Pathway categories:",
    gdsc_drug_pharmacology[
        "PATHWAY_NAME"
    ].nunique(),
)

Eligible drugs: 281
FDR-associated drugs: 222
Drugs with >=1 parsed target: 242
Drugs without usable target annotation: 39
Pathway categories: 24


In [58]:
# =============================================================================
# Consolidate frozen Phase 6 drug-eligibility metadata across resources
# =============================================================================

gdsc_eligibility_handoff = (
    gdsc_eligibility
    .rename(
        columns={
            "DRUG_ID": "resource_drug_id",
        }
    )
    .assign(
        analysis_universe="native_gdsc",
    )
)

ctrp_eligibility_handoff = (
    ctrp_eligibility
    .rename(
        columns={
            "master_cpd_id": "resource_drug_id",
        }
    )
    .assign(
        analysis_universe="native_ctrp",
    )
)

prism_eligibility_handoff = (
    prism_eligibility
    .rename(
        columns={
            "broad_id": "resource_drug_id",
        }
    )
    .assign(
        analysis_universe="exact_cross_resource_prism",
    )
)

phase6_drug_eligibility = pd.concat(
    [
        gdsc_eligibility_handoff,
        ctrp_eligibility_handoff,
        prism_eligibility_handoff,
    ],
    ignore_index=True,
)

phase6_drug_eligibility[
    "resource_drug_id"
] = (
    phase6_drug_eligibility[
        "resource_drug_id"
    ]
    .astype(str)
)

assert len(phase6_drug_eligibility) == (
    len(gdsc_eligibility)
    + len(ctrp_eligibility)
    + len(prism_eligibility)
)

print(
    "Drug-eligibility rows:",
    len(phase6_drug_eligibility),
)

display(
    phase6_drug_eligibility
    .groupby(
        [
            "resource",
            "analysis_universe",
        ],
        observed=True,
    )
    .agg(
        drugs_evaluated=(
            "resource_drug_id",
            "nunique",
        ),
        drugs_eligible=(
            "eligible",
            "sum",
        ),
    )
    .reset_index()
)

Drug-eligibility rows: 1037


,resource,analysis_universe,drugs_evaluated,drugs_eligible
0,CTRP,native_ctrp,545,499
1,GDSC,native_gdsc,295,281
2,PRISM_exact_cross_resource,exact_cross_resource_prism,197,194


In [59]:
# =============================================================================
# Consolidate frozen exact cross-resource compound catalog
# =============================================================================

prism_assignment_context = (
    prism_cross_resource_compounds[
        [
            "drug_name_key",
            "prism_broad_id",
            "screen_pattern",
            "primary_prism_screen",
        ]
    ]
    .drop_duplicates()
)

phase6_cross_resource_compounds = (
    cross_resource_drug_coverage
    .merge(
        prism_assignment_context,
        on=[
            "drug_name_key",
            "prism_broad_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "resources_present",
            "drug_name_key",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

assert len(phase6_cross_resource_compounds) == 228
assert (
    phase6_cross_resource_compounds[
        "drug_name_key"
    ].nunique()
    == 228
)

print(
    "Exact cross-resource compounds:",
    len(phase6_cross_resource_compounds),
)
print(
    "Present in all 3 resources:",
    (
        phase6_cross_resource_compounds[
            "resources_present"
        ] == 3
    ).sum(),
)
print(
    "Present in exactly 2 resources:",
    (
        phase6_cross_resource_compounds[
            "resources_present"
        ] == 2
    ).sum(),
)
print(
    "Represented in PRISM:",
    phase6_cross_resource_compounds[
        "prism_broad_id"
    ].notna().sum(),
)
print(
    "Unique primary PRISM assignment:",
    phase6_cross_resource_compounds[
        "primary_prism_screen"
    ]
    .isin(
        [
            "HTS002",
            "MTS006",
            "MTS010",
        ]
    )
    .sum(),
)
print(
    "AMBIGUOUS_NONREDO:",
    (
        phase6_cross_resource_compounds[
            "primary_prism_screen"
        ]
        == "AMBIGUOUS_NONREDO"
    ).sum(),
)
print(
    "No PRISM representation:",
    phase6_cross_resource_compounds[
        "prism_broad_id"
    ].isna().sum(),
)

display(
    phase6_cross_resource_compounds.head()
)

Exact cross-resource compounds: 228
Present in all 3 resources: 39
Present in exactly 2 resources: 189
Represented in PRISM: 199
Unique primary PRISM assignment: 197
AMBIGUOUS_NONREDO: 2
No PRISM representation: 29


,drug_name_key,gdsc_drug_id,gdsc_drug_name,ctrp_master_cpd_id,ctrp_drug_name,prism_broad_id,prism_drug_name,resources_present,gdsc_models,ctrp_models,prism_HTS002_models,prism_MTS005_models,prism_MTS006_models,prism_MTS010_models,screen_pattern,primary_prism_screen
0,ABT737,1910.0,ABT737,411738.0,ABT-737,BRD-K56301217-001-07-4,ABT-737,3,703,708,452,0,0,0,HTS002,HTS002
1,ALISERTIB,1051.0,Alisertib,636711.0,alisertib,BRD-K75295174-001-05-0,alisertib,3,697,774,471,0,0,0,HTS002,HTS002
2,AXITINIB,1021.0,Axitinib,348990.0,axitinib,BRD-K29905972-001-06-3,axitinib,3,706,773,437,0,0,430,HTS002+MTS010,MTS010
3,AZD4547,1786.0,AZD4547,660325.0,AZD4547,BRD-K28392481-001-05-1,AZD4547,3,706,702,344,0,0,0,HTS002,HTS002
4,AZD6482,2169.0,AZD6482,639390.0,AZD6482,BRD-K58772419-001-07-0,AZD6482,3,175,785,473,0,0,0,HTS002,HTS002


In [60]:
# =============================================================================
# Consolidate frozen CTRP-to-DepMap Phase 6 model crosswalk
# =============================================================================

phase6_ctrp_model_crosswalk = (
    ctrp_phase6_crosswalk
    .merge(
        phase6_score_universe[
            [
                "ModelID",
                "OncotreeLineage",
                "score_origin",
            ]
        ],
        on="ModelID",
        how="left",
        validate="one_to_one",
    )
    .sort_values("ModelID")
    .reset_index(drop=True)
)

assert len(phase6_ctrp_model_crosswalk) == 821

assert (
    phase6_ctrp_model_crosswalk[
        "ModelID"
    ].nunique()
    == 821
)

assert (
    phase6_ctrp_model_crosswalk[
        "master_ccl_id"
    ].nunique()
    == 821
)

assert phase6_ctrp_model_crosswalk[
    [
        "OncotreeLineage",
        "score_origin",
    ]
].notna().all().all()

print(
    "CTRP Phase 6 mapped models:",
    len(phase6_ctrp_model_crosswalk),
)
print(
    "Frozen Phase 4 anchor models:",
    (
        phase6_ctrp_model_crosswalk[
            "score_origin"
        ]
        == "frozen_phase4"
    ).sum(),
)
print(
    "Projected external models:",
    (
        phase6_ctrp_model_crosswalk[
            "score_origin"
        ]
        == "projected_from_frozen_phase4"
    ).sum(),
)
print(
    "Represented lineages:",
    phase6_ctrp_model_crosswalk[
        "OncotreeLineage"
    ].nunique(),
)
print(
    "Mapping methods:",
    phase6_ctrp_model_crosswalk[
        "mapping_method"
    ].nunique(),
)

display(
    phase6_ctrp_model_crosswalk.head()
)

CTRP Phase 6 mapped models: 821
Frozen Phase 4 anchor models: 565
Projected external models: 256
Represented lineages: 25
Mapping methods: 1


,master_ccl_id,ccl_name,ModelID,CellLineName,mapping_method,OncotreeLineage,score_origin
0,371,HL60,ACH-000002,HL-60,exact_normalized_name_globally_unambiguous,Myeloid,frozen_phase4
1,362,HEL,ACH-000004,HEL,exact_normalized_name_globally_unambiguous,Myeloid,frozen_phase4
2,363,HEL9217,ACH-000005,HEL 92.1.7,exact_normalized_name_globally_unambiguous,Myeloid,projected_from_frozen_phase4
3,702,MONOMAC6,ACH-000006,MONO-MAC-6,exact_normalized_name_globally_unambiguous,Myeloid,frozen_phase4
4,632,LS513,ACH-000007,LS513,exact_normalized_name_globally_unambiguous,Bowel,frozen_phase4


In [61]:
# =============================================================================
# Consolidate frozen Phase 6 program-score universe
# =============================================================================

phase6_program_score_universe = (
    phase6_score_universe[
        [
            "ModelID",
            "OncotreeLineage",
            "CONSENSUS_TX_01",
            "CONSENSUS_TX_02",
            "CONSENSUS_TX_03",
            "score_origin",
        ]
    ]
    .sort_values("ModelID")
    .reset_index(drop=True)
)

assert len(phase6_program_score_universe) == 981

assert (
    phase6_program_score_universe[
        "ModelID"
    ].nunique()
    == 981
)

assert phase6_program_score_universe[
    [
        "ModelID",
        "OncotreeLineage",
        "CONSENSUS_TX_01",
        "CONSENSUS_TX_02",
        "CONSENSUS_TX_03",
        "score_origin",
    ]
].notna().all().all()

assert set(
    phase6_program_score_universe[
        "score_origin"
    ].unique()
) == {
    "frozen_phase4",
    "projected_from_frozen_phase4",
}

print(
    "Phase 6 score universe:",
    len(phase6_program_score_universe),
)
print(
    "Frozen Phase 4 models:",
    (
        phase6_program_score_universe[
            "score_origin"
        ]
        == "frozen_phase4"
    ).sum(),
)
print(
    "Projected external models:",
    (
        phase6_program_score_universe[
            "score_origin"
        ]
        == "projected_from_frozen_phase4"
    ).sum(),
)
print(
    "Represented lineages:",
    phase6_program_score_universe[
        "OncotreeLineage"
    ].nunique(),
)

display(
    phase6_program_score_universe.head()
)

Phase 6 score universe: 981
Frozen Phase 4 models: 713
Projected external models: 268
Represented lineages: 28


,ModelID,OncotreeLineage,CONSENSUS_TX_01,CONSENSUS_TX_02,CONSENSUS_TX_03,score_origin
0,ACH-000001,Ovary/Fallopian Tube,-0.618908,0.121723,-0.047979,frozen_phase4
1,ACH-000002,Myeloid,1.312391,0.062750,-0.407409,frozen_phase4
2,ACH-000004,Myeloid,1.236019,0.613855,-0.498685,frozen_phase4
3,ACH-000005,Myeloid,1.163084,0.709155,-0.574934,projected_from_frozen_phase4
4,ACH-000006,Myeloid,1.496321,0.404348,-0.515095,frozen_phase4


In [62]:
# =============================================================================
# Reconcile lineage coverage between frozen and projected score origins
# =============================================================================

frozen_lineages = set(
    phase6_program_score_universe.loc[
        phase6_program_score_universe[
            "score_origin"
        ] == "frozen_phase4",
        "OncotreeLineage",
    ]
)

projected_lineages = set(
    phase6_program_score_universe.loc[
        phase6_program_score_universe[
            "score_origin"
        ] == "projected_from_frozen_phase4",
        "OncotreeLineage",
    ]
)

print(
    "Frozen Phase 4 lineages:",
    len(frozen_lineages),
)
print(
    "Projected external lineages:",
    len(projected_lineages),
)
print(
    "Union lineages:",
    len(
        frozen_lineages
        | projected_lineages
    ),
)

print(
    "\nProjected-only lineages:",
    sorted(
        projected_lineages
        - frozen_lineages
    ),
)

print(
    "\nFrozen-only lineages:",
    sorted(
        frozen_lineages
        - projected_lineages
    ),
)

Frozen Phase 4 lineages: 27
Projected external lineages: 23
Union lineages: 28

Projected-only lineages: ['Ampulla of Vater']

Frozen-only lineages: ['Adrenal Gland', 'Cervix', 'Prostate', 'Testis', 'Vulva/Vagina']


In [63]:
# =============================================================================
# Consolidate lineage-supported primary GDSC analysis universe
# =============================================================================

phase6_gdsc_analysis_universe = (
    gdsc_primary_universe[
        [
            "DRUG_ID",
            "ModelID",
            "OncotreeLineage",
            "response_value",
            "response_metric",
        ]
    ]
    .assign(
        resource="GDSC",
        evidence_role="developmental_internal",
    )
    .sort_values(
        [
            "DRUG_ID",
            "ModelID",
        ]
    )
    .reset_index(drop=True)
)

# The handoff must preserve the previously established primary universe.
assert len(phase6_gdsc_analysis_universe) == len(
    gdsc_primary_universe
)

assert not phase6_gdsc_analysis_universe[
    ["DRUG_ID", "ModelID"]
].duplicated().any()

assert phase6_gdsc_analysis_universe[
    "response_metric"
].eq("LN_IC50").all()

print(
    "Stable GDSC analysis universe:",
    phase6_gdsc_analysis_universe.shape,
)

display(
    phase6_gdsc_analysis_universe.head()
)

Stable GDSC analysis universe: (136176, 7)


,DRUG_ID,ModelID,OncotreeLineage,response_value,response_metric,resource,evidence_role
0,1003,ACH-000001,Ovary/Fallopian Tube,-1.179383,LN_IC50,GDSC,developmental_internal
1,1003,ACH-000002,Myeloid,-4.472633,LN_IC50,GDSC,developmental_internal
2,1003,ACH-000004,Myeloid,-3.978342,LN_IC50,GDSC,developmental_internal
3,1003,ACH-000006,Myeloid,-3.878637,LN_IC50,GDSC,developmental_internal
4,1003,ACH-000007,Bowel,-3.160381,LN_IC50,GDSC,developmental_internal


In [64]:
# =============================================================================
# Consolidate lineage-supported primary CTRP analysis universe
# =============================================================================

phase6_ctrp_analysis_universe = (
    ctrp_primary_universe[
        [
            "master_cpd_id",
            "ModelID",
            "OncotreeLineage",
            "response_value",
            "n_experiments",
            "response_metric",
        ]
    ]
    .assign(
        resource="CTRP",
        evidence_role="external_replication",
    )
    .sort_values(
        [
            "master_cpd_id",
            "ModelID",
        ]
    )
    .reset_index(drop=True)
)

# The handoff must preserve the previously established primary universe.
assert len(phase6_ctrp_analysis_universe) == len(
    ctrp_primary_universe
)

assert not phase6_ctrp_analysis_universe[
    ["master_cpd_id", "ModelID"]
].duplicated().any()

assert phase6_ctrp_analysis_universe[
    "response_metric"
].eq("area_under_curve").all()

assert phase6_ctrp_analysis_universe[
    "n_experiments"
].ge(1).all()

print(
    "Stable CTRP analysis universe:",
    phase6_ctrp_analysis_universe.shape,
)

display(
    phase6_ctrp_analysis_universe.head()
)

Stable CTRP analysis universe: (302310, 8)


,master_cpd_id,ModelID,OncotreeLineage,response_value,n_experiments,response_metric,resource,evidence_role
0,1788,ACH-000007,Bowel,14.981,1,area_under_curve,CTRP,external_replication
1,1788,ACH-000012,Lung,14.201,1,area_under_curve,CTRP,external_replication
2,1788,ACH-000030,Lung,16.316,1,area_under_curve,CTRP,external_replication
3,1788,ACH-000035,Lung,13.838,1,area_under_curve,CTRP,external_replication
4,1788,ACH-000050,Lymphoid,14.802,1,area_under_curve,CTRP,external_replication


In [65]:
# =============================================================================
# Consolidate lineage-supported primary PRISM analysis universe
# =============================================================================

phase6_prism_analysis_universe = (
    prism_primary_universe[
        [
            "broad_id",
            "ModelID",
            "OncotreeLineage",
            "screen_id",
            "primary_prism_screen",
            "response_value",
            "response_metric",
        ]
    ]
    .assign(
        resource="PRISM",
        evidence_role="external_replication",
    )
    .sort_values(
        [
            "broad_id",
            "ModelID",
        ]
    )
    .reset_index(drop=True)
)

# The handoff must preserve the previously established primary universe.
assert len(phase6_prism_analysis_universe) == len(
    prism_primary_universe
)

assert not phase6_prism_analysis_universe[
    ["broad_id", "ModelID"]
].duplicated().any()

assert phase6_prism_analysis_universe[
    "response_metric"
].eq("auc").all()

assert (
    phase6_prism_analysis_universe["screen_id"]
    == phase6_prism_analysis_universe["primary_prism_screen"]
).all()

print(
    "Stable PRISM analysis universe:",
    phase6_prism_analysis_universe.shape,
)

display(
    phase6_prism_analysis_universe.head()
)

Stable PRISM analysis universe: (65736, 9)


,broad_id,ModelID,OncotreeLineage,screen_id,primary_prism_screen,response_value,response_metric,resource,evidence_role
0,BRD-A09722536-002-18-0,ACH-000008,Skin,HTS002,HTS002,0.969551,auc,PRISM,external_replication
1,BRD-A09722536-002-18-0,ACH-000012,Lung,HTS002,HTS002,0.981941,auc,PRISM,external_replication
2,BRD-A09722536-002-18-0,ACH-000013,Ovary/Fallopian Tube,HTS002,HTS002,1.010348,auc,PRISM,external_replication
3,BRD-A09722536-002-18-0,ACH-000014,Skin,HTS002,HTS002,0.995421,auc,PRISM,external_replication
4,BRD-A09722536-002-18-0,ACH-000021,Lung,HTS002,HTS002,1.080709,auc,PRISM,external_replication


In [66]:
# =============================================================================
# Consolidate complete GDSC program–drug association handoff
# =============================================================================

gdsc_drug_context_handoff = (
    gdsc_drug_pharmacology[
        [
            "DRUG_ID",
            "DRUG_NAME",
            "PATHWAY_NAME",
            "PUTATIVE_TARGET",
            "parsed_target_count",
            "parsed_targets",
        ]
    ]
    .copy()
)

phase6_gdsc_program_drug_associations = (
    gdsc_primary_associations_with_lolo
    .merge(
        gdsc_drug_context_handoff,
        on="DRUG_ID",
        how="left",
        validate="many_to_one",
    )
    .assign(
        association_direction=lambda data: np.where(
            data["beta"] > 0,
            "resistance_like",
            "sensitivity_like",
        )
    )
    .sort_values(
        [
            "DRUG_ID",
            "program",
        ]
    )
    .reset_index(drop=True)
)

# This handoff adds pharmacological context without changing the
# previously established association universe.
assert len(phase6_gdsc_program_drug_associations) == len(
    gdsc_primary_associations_with_lolo
)

assert not phase6_gdsc_program_drug_associations[
    ["DRUG_ID", "program"]
].duplicated().any()

assert phase6_gdsc_program_drug_associations[
    [
        "DRUG_NAME",
        "PATHWAY_NAME",
        "parsed_target_count",
    ]
].notna().all().all()

assert set(
    phase6_gdsc_program_drug_associations[
        "association_direction"
    ]
) <= {
    "resistance_like",
    "sensitivity_like",
}

print(
    "Stable GDSC association handoff:",
    phase6_gdsc_program_drug_associations.shape,
)

display(
    phase6_gdsc_program_drug_associations.head()
)

Stable GDSC association handoff: (843, 30)


,resource,evidence_role,DRUG_ID,program,response_metric,n_models,n_lineages,beta,se_hc3,ci95_low,...,lolo_same_sign_fraction,lolo_any_sign_reversal,lolo_max_abs_beta_change,lolo_max_change_omitted_lineage,DRUG_NAME,PATHWAY_NAME,PUTATIVE_TARGET,parsed_target_count,parsed_targets,association_direction
0,GDSC,developmental_internal,1003,CONSENSUS_TX_01,LN_IC50,556,11,-0.752874,0.185047,-1.116368,...,1.000000,False,0.903753,Lymphoid,Camptothecin,DNA replication,TOP1,1,TOP1,sensitivity_like
1,GDSC,developmental_internal,1003,CONSENSUS_TX_02,LN_IC50,556,11,-0.339577,0.106594,-0.548964,...,1.000000,False,0.116642,Lung,Camptothecin,DNA replication,TOP1,1,TOP1,sensitivity_like
2,GDSC,developmental_internal,1003,CONSENSUS_TX_03,LN_IC50,556,11,0.044303,0.120183,-0.191776,...,0.818182,True,0.095843,CNS/Brain,Camptothecin,DNA replication,TOP1,1,TOP1,resistance_like
3,GDSC,developmental_internal,1004,CONSENSUS_TX_01,LN_IC50,419,10,-0.696002,0.217456,-1.123475,...,1.000000,False,1.231325,Lymphoid,Vinblastine,Mitosis,Microtubule destabiliser,1,Microtubule destabiliser,sensitivity_like
4,GDSC,developmental_internal,1004,CONSENSUS_TX_02,LN_IC50,419,10,-0.872190,0.173276,-1.212815,...,1.000000,False,0.111548,Esophagus/Stomach,Vinblastine,Mitosis,Microtubule destabiliser,1,Microtubule destabiliser,sensitivity_like


In [67]:
# =============================================================================
# Summarize stable notebook-600 analysis objects
# =============================================================================

frozen_score_mask = phase6_program_score_universe[
    "score_origin"
].eq("frozen_phase4")

projected_score_mask = phase6_program_score_universe[
    "score_origin"
].eq("projected_from_frozen_phase4")

frozen_lineages = set(
    phase6_program_score_universe.loc[
        frozen_score_mask,
        "OncotreeLineage",
    ]
)

projected_lineages = set(
    phase6_program_score_universe.loc[
        projected_score_mask,
        "OncotreeLineage",
    ]
)

score_universe_metadata = {
    "models_total": int(len(phase6_program_score_universe)),
    "frozen_phase4_models": int(frozen_score_mask.sum()),
    "projected_external_models": int(projected_score_mask.sum()),
    "lineages_union": int(
        phase6_program_score_universe["OncotreeLineage"].nunique()
    ),
    "lineages_frozen_phase4": len(frozen_lineages),
    "lineages_projected_external": len(projected_lineages),
    "projected_only_lineages": sorted(
        projected_lineages - frozen_lineages
    ),
    "external_projection_rule": (
        "Frozen Phase 4 gene-standardization parameters, "
        "consensus weights, and score-standardization parameters; "
        "no external or combined-cohort re-standardization."
    ),
}

identity_harmonization_metadata = {
    "ctrp_mapped_models": int(len(phase6_ctrp_model_crosswalk)),
    "ctrp_anchor_models": int(
        phase6_ctrp_model_crosswalk["score_origin"]
        .eq("frozen_phase4")
        .sum()
    ),
    "ctrp_projected_external_models": int(
        phase6_ctrp_model_crosswalk["score_origin"]
        .eq("projected_from_frozen_phase4")
        .sum()
    ),
    "ctrp_mapping_method": "exact_normalized_name_globally_unambiguous",
    "ctrp_globally_ambiguous_excluded": ["KMH2"],
    "cross_resource_exact_compounds": int(
        len(phase6_cross_resource_compounds)
    ),
    "cross_resource_compounds_all_three": int(
        phase6_cross_resource_compounds["resources_present"].eq(3).sum()
    ),
    "cross_resource_compounds_exactly_two": int(
        phase6_cross_resource_compounds["resources_present"].eq(2).sum()
    ),
    "compound_matching_rule": (
        "Unambiguous normalized compound-name key within each "
        "participating resource; no fuzzy, target-based, "
        "mechanism-based, or manual rescue."
    ),
}


def summarize_analysis_universe(data, drug_column):
    return {
        "observations": int(len(data)),
        "drugs": int(data[drug_column].nunique()),
        "models": int(data["ModelID"].nunique()),
        "supported_lineages": int(
            data["OncotreeLineage"].nunique()
        ),
    }


analysis_universe_metadata = {
    "GDSC": summarize_analysis_universe(
        phase6_gdsc_analysis_universe,
        "DRUG_ID",
    ),
    "CTRP": {
        **summarize_analysis_universe(
            phase6_ctrp_analysis_universe,
            "master_cpd_id",
        ),
        "single_experiment_pairs": int(
            phase6_ctrp_analysis_universe["n_experiments"].eq(1).sum()
        ),
        "repeated_experiment_pairs": int(
            phase6_ctrp_analysis_universe["n_experiments"].gt(1).sum()
        ),
        "maximum_experiments_per_pair": int(
            phase6_ctrp_analysis_universe["n_experiments"].max()
        ),
    },
    "PRISM": {
        **summarize_analysis_universe(
            phase6_prism_analysis_universe,
            "broad_id",
        ),
        "eligible_compounds_by_screen": {
            screen: int(count)
            for screen, count in (
                phase6_prism_analysis_universe[
                    ["broad_id", "screen_id"]
                ]
                .drop_duplicates()["screen_id"]
                .value_counts()
                .sort_index()
                .items()
            )
        },
    },
}

eligibility_rule_metadata = {
    "minimum_models_per_supported_lineage": 20,
    "minimum_supported_lineages": 3,
    "minimum_models_across_supported_lineages": 100,
    "resources": {
        "GDSC": {
            "evaluated": int(len(gdsc_eligibility)),
            "eligible": int(gdsc_eligibility["eligible"].sum()),
        },
        "CTRP": {
            "evaluated": int(len(ctrp_eligibility)),
            "eligible": int(ctrp_eligibility["eligible"].sum()),
        },
        "PRISM_exact_cross_resource": {
            "evaluated": int(len(prism_eligibility)),
            "eligible": int(prism_eligibility["eligible"].sum()),
        },
    },
}

prism_screen_assignment_metadata = {
    "cross_resource_compounds_with_prism": int(
        phase6_cross_resource_compounds["prism_broad_id"].notna().sum()
    ),
    "unique_primary_screen_assignment": int(
        phase6_cross_resource_compounds["primary_prism_screen"]
        .isin(["HTS002", "MTS006", "MTS010"])
        .sum()
    ),
    "ambiguous_nonredo": int(
        phase6_cross_resource_compounds["primary_prism_screen"]
        .eq("AMBIGUOUS_NONREDO")
        .sum()
    ),
}

pharmacologic_context_metadata = {
    "eligible_gdsc_drugs": int(
        gdsc_drug_pharmacology["DRUG_ID"].nunique()
    ),
    "drugs_with_usable_provider_target": int(
        gdsc_drug_pharmacology.loc[
            gdsc_drug_pharmacology["parsed_target_count"].gt(0),
            "DRUG_ID",
        ].nunique()
    ),
    "drugs_without_usable_provider_target": int(
        gdsc_drug_pharmacology.loc[
            gdsc_drug_pharmacology["parsed_target_count"].eq(0),
            "DRUG_ID",
        ].nunique()
    ),
    "pathway_categories": int(
        gdsc_drug_pharmacology["PATHWAY_NAME"].nunique()
    ),
    "target_parsing_rule": (
        "Split provider PUTATIVE_TARGET only on commas; trim "
        "whitespace; preserve punctuation including '/'; no "
        "synonym mapping, gene harmonization, manual curation, "
        "or target imputation."
    ),
    "target_and_pathway_role": (
        "Descriptive pharmacological context only; not chemical "
        "drug-family identity and not used to modify FDR results."
    ),
}

In [68]:
# =============================================================================
# Define notebook-600 methodological metadata
# =============================================================================

response_representation_metadata = {
    "GDSC": {
        "metric": "LN_IC50",
        "higher_is": "more_resistance_like",
    },
    "CTRP": {
        "metric": "area_under_curve",
        "higher_is": "more_resistance_like",
        "repeated_pair_primary_aggregation": "median",
    },
    "PRISM": {
        "metric": "auc",
        "higher_is": "more_resistance_like",
        "screen_rule": (
            "MTS010 when available; otherwise unique non-redo "
            "screen; multiple non-redo screens without MTS010 "
            "remain AMBIGUOUS_NONREDO."
        ),
    },
    "cross_resource_scale_policy": (
        "Resource-specific response values are not pooled or "
        "treated as numerically exchangeable."
    ),
}

primary_gdsc_inference_metadata = {
    "formula": "LN_IC50 ~ program_score + C(OncotreeLineage)",
    "programs_fitted_separately": True,
    "estimator": "ordinary_least_squares",
    "robust_covariance": "HC3",
    "two_sided_t_inference": True,
    "use_t": True,
    "confidence_interval": 0.95,
    "primary_tests": int(
        len(phase6_gdsc_program_drug_associations)
    ),
    "multiplicity_method": "Benjamini-Hochberg",
    "multiplicity_family": "all_843_GDSC_program_drug_tests",
    "fdr_threshold": 0.05,
    "fdr_associations": int(
        phase6_gdsc_program_drug_associations["fdr_05"].sum()
    ),
    "fdr_associated_unique_drugs": int(
        phase6_gdsc_program_drug_associations.loc[
            phase6_gdsc_program_drug_associations["fdr_05"],
            "DRUG_ID",
        ].nunique()
    ),
    "hard_effect_size_threshold": None,
}

lolo_characterization_metadata = {
    "role": "descriptive_lineage_composition_sensitivity",
    "significance_retesting": False,
    "associations_with_any_sign_reversal": int(
        phase6_gdsc_program_drug_associations[
            "lolo_any_sign_reversal"
        ].sum()
    ),
    "fdr_associations_with_any_sign_reversal": int(
        (
            phase6_gdsc_program_drug_associations["fdr_05"]
            & phase6_gdsc_program_drug_associations[
                "lolo_any_sign_reversal"
            ]
        ).sum()
    ),
    "timing": (
        "Exact LOLO descriptive summaries were frozen after "
        "primary GDSC inference but before inspection of any "
        "LOLO refit result."
    ),
}

known_limitations_metadata = [
    (
        "GDSC is developmental/internal rather than independent "
        "replication because of upstream GDSC-linked construction."
    ),
    (
        "No defensible frozen proliferation covariate was available; "
        "residual proliferation confounding remains possible."
    ),
    (
        "Cross-resource normalized-name matches are computational "
        "candidate exact-compound links and do not independently "
        "establish chemical identity, salt, stereochemistry, or "
        "formulation equivalence."
    ),
    (
        "CTRP and PRISM external association/replication results "
        "have not been inspected in notebook 600."
    ),
    (
        "Target and pathway annotations are descriptive context and "
        "must not be treated as formal drug-family identifiers."
    ),
    (
        "Response magnitudes are resource-specific and must not be "
        "naively compared or pooled across GDSC, CTRP, and PRISM."
    ),
]

stable_interfaces_metadata = [
    "phase6.600.program_score_universe",
    "phase6.600.ctrp_model_crosswalk",
    "phase6.600.cross_resource_compound_catalog",
    "phase6.600.drug_eligibility",
    "phase6.600.gdsc_analysis_universe",
    "phase6.600.ctrp_analysis_universe",
    "phase6.600.prism_analysis_universe",
    "phase6.600.gdsc_program_drug_associations",
    "phase6.600.analysis_metadata",
]

In [69]:
# =============================================================================
# Assemble deterministic notebook-600 analysis metadata
# =============================================================================

import json

phase6_600_analysis_metadata = {
    "schema_version": 1,
    "phase": 6,
    "producer_notebook": (
        "notebooks/phase6_pharmacogenomic_contexts/"
        "600_program_drug_associations.ipynb"
    ),
    "analysis_contract": (
        "docs/contracts/phase6/PHASE6_ANALYSIS_CONTRACT.md"
    ),
    "scientific_scope": (
        "Program-level pharmacogenomic association characterization and "
        "preparation of stable cross-screen handoff objects."
    ),
    "programs": [
        "CONSENSUS_TX_01",
        "CONSENSUS_TX_02",
        "CONSENSUS_TX_03",
    ],
    "resource_roles": {
        "GDSC": "developmental_internal",
        "CTRP": "external_replication",
        "PRISM": "external_replication",
    },
    "score_universe": score_universe_metadata,
    "identity_harmonization": identity_harmonization_metadata,
    "response_representations": response_representation_metadata,
    "eligibility_rule": eligibility_rule_metadata,
    "analysis_universes": analysis_universe_metadata,
    "primary_gdsc_inference": primary_gdsc_inference_metadata,
    "lolo_characterization": lolo_characterization_metadata,
    "prism_primary_screen_assignment": prism_screen_assignment_metadata,
    "pharmacologic_context": pharmacologic_context_metadata,
    "known_limitations": known_limitations_metadata,
    "stable_interfaces": stable_interfaces_metadata,
}

# Serialization is the only local validity requirement at this stage.
json.dumps(
    phase6_600_analysis_metadata,
    indent=2,
    sort_keys=True,
)

print(
    "Metadata sections:",
    len(phase6_600_analysis_metadata),
)
print(
    "Stable interfaces:",
    len(stable_interfaces_metadata),
)

Metadata sections: 18
Stable interfaces: 9


In [70]:
# =============================================================================
# Prepare stable notebook-600 output representations
# =============================================================================

import hashlib
import json

phase6_output_dir = (
    Paths.root
    / "data"
    / "processed"
    / "pharmacogenomic_contexts"
)

phase6_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# Nullable integers preserve native identifiers when cross-resource
# coverage produces missing values.
cross_resource_output = (
    phase6_cross_resource_compounds
    .copy()
)

for column in [
    "gdsc_drug_id",
    "ctrp_master_cpd_id",
]:
    cross_resource_output[column] = (
        cross_resource_output[column]
        .astype("Int64")
    )

drug_eligibility_output = (
    phase6_drug_eligibility
    .sort_values(
        [
            "resource",
            "resource_drug_id",
        ]
    )
    .reset_index(drop=True)
)


def serialize_target_tokens(value):
    if isinstance(value, (list, tuple)):
        return json.dumps(
            list(value),
            ensure_ascii=False,
            separators=(",", ":"),
        )

    if value is None:
        return None

    if isinstance(value, float) and np.isnan(value):
        return None

    return str(value)


# Parsed target collections are serialized explicitly to avoid
# implementation-dependent object-column persistence.
gdsc_association_output = (
    phase6_gdsc_program_drug_associations
    .copy()
)

gdsc_association_output[
    "parsed_targets"
] = (
    gdsc_association_output[
        "parsed_targets"
    ]
    .map(serialize_target_tokens)
)

In [71]:
# =============================================================================
# Define stable notebook-600 artifact payloads
# =============================================================================

def artifact_payload(
    filename,
    data,
    artifact_format,
):
    return {
        "path": phase6_output_dir / filename,
        "data": data,
        "format": artifact_format,
    }


artifact_payloads = {
    "phase6.600.program_score_universe": artifact_payload(
        "600_program_score_universe.parquet",
        phase6_program_score_universe,
        "parquet",
    ),
    "phase6.600.ctrp_model_crosswalk": artifact_payload(
        "600_ctrp_model_crosswalk.csv",
        phase6_ctrp_model_crosswalk,
        "csv",
    ),
    "phase6.600.cross_resource_compound_catalog": artifact_payload(
        "600_cross_resource_exact_compound_catalog.csv",
        cross_resource_output,
        "csv",
    ),
    "phase6.600.drug_eligibility": artifact_payload(
        "600_drug_eligibility.csv",
        drug_eligibility_output,
        "csv",
    ),
    "phase6.600.gdsc_analysis_universe": artifact_payload(
        "600_gdsc_analysis_universe.parquet",
        phase6_gdsc_analysis_universe,
        "parquet",
    ),
    "phase6.600.ctrp_analysis_universe": artifact_payload(
        "600_ctrp_analysis_universe.parquet",
        phase6_ctrp_analysis_universe,
        "parquet",
    ),
    "phase6.600.prism_analysis_universe": artifact_payload(
        "600_prism_analysis_universe.parquet",
        phase6_prism_analysis_universe,
        "parquet",
    ),
    "phase6.600.gdsc_program_drug_associations": artifact_payload(
        "600_gdsc_program_drug_associations.parquet",
        gdsc_association_output,
        "parquet",
    ),
    "phase6.600.analysis_metadata": artifact_payload(
        "600_program_drug_association_metadata.json",
        phase6_600_analysis_metadata,
        "json",
    ),
}

assert set(artifact_payloads) == set(
    phase6_600_analysis_metadata[
        "stable_interfaces"
    ]
)

In [72]:
# =============================================================================
# Persist and fingerprint stable notebook-600 artifacts
# =============================================================================

def write_artifact(artifact):
    path = artifact["path"]
    data = artifact["data"]
    artifact_format = artifact["format"]

    if artifact_format == "parquet":
        data.to_parquet(
            path,
            index=False,
        )

    elif artifact_format == "csv":
        data.to_csv(
            path,
            index=False,
            encoding="utf-8",
            lineterminator="\n",
        )

    elif artifact_format == "json":
        with path.open(
            "w",
            encoding="utf-8",
            newline="\n",
        ) as handle:
            json.dump(
                data,
                handle,
                ensure_ascii=False,
                indent=2,
                sort_keys=True,
            )
            handle.write("\n")

    else:
        raise ValueError(
            f"Unsupported artifact format: "
            f"{artifact_format}"
        )


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


for artifact in artifact_payloads.values():
    write_artifact(artifact)


artifact_manifest_rows = []

for artifact_id, artifact in artifact_payloads.items():
    path = artifact["path"]
    data = artifact["data"]

    assert path.exists()
    assert path.stat().st_size > 0

    rows, columns = (
        data.shape
        if isinstance(data, pd.DataFrame)
        else (None, None)
    )

    artifact_manifest_rows.append(
        {
            "artifact_id": artifact_id,
            "path": str(
                path.relative_to(Paths.root)
            ).replace("\\", "/"),
            "rows": rows,
            "columns": columns,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )


phase6_600_artifact_manifest = (
    pd.DataFrame(artifact_manifest_rows)
    .sort_values("artifact_id")
    .reset_index(drop=True)
)

display(
    phase6_600_artifact_manifest
)

,artifact_id,path,rows,columns,size_bytes,sha256
0,phase6.600.analysis_metadata,data/processed/pharmacogenomic_contexts/600_pr...,NaN,NaN,6223,030bcf6000272aa6df44f2321c2a5c098cbb7a1bb4923f...
1,phase6.600.cross_resource_compound_catalog,data/processed/pharmacogenomic_contexts/600_cr...,228.0,16.0,21746,ab3b5f12893224f2d0273ef234b2bb8c5cc13143fe4fb2...
2,phase6.600.ctrp_analysis_universe,data/processed/pharmacogenomic_contexts/600_ct...,302310.0,8.0,1096011,a1c92ef3c99f3651f8013196ecbe64896e17f946e2733c...
3,phase6.600.ctrp_model_crosswalk,data/processed/pharmacogenomic_contexts/600_ct...,821.0,7.0,83078,3e9530416fc9294af4e0ef52ab6cbcac384002e2013a39...
4,phase6.600.drug_eligibility,data/processed/pharmacogenomic_contexts/600_dr...,1037.0,6.0,47011,a4fee84775de46c0d7ef5ae510972c167effe2cee072ed...
5,phase6.600.gdsc_analysis_universe,data/processed/pharmacogenomic_contexts/600_gd...,136176.0,7.0,1430223,00954d16b2082f3d660a9bf4f58947f5175aa3483667d1...
6,phase6.600.gdsc_program_drug_associations,data/processed/pharmacogenomic_contexts/600_gd...,843.0,30.0,118162,44bebf416698a02f07889a6c07a430eabba53f15e6887b...
7,phase6.600.prism_analysis_universe,data/processed/pharmacogenomic_contexts/600_pr...,65736.0,9.0,703400,f27a5fa4c9d7674d7efec7e9e22a0aab1334b512e1c73e...
8,phase6.600.program_score_universe,data/processed/pharmacogenomic_contexts/600_pr...,981.0,6.0,38838,1ce2cba61373d601dfad1a0c188550645ceda1c1a94dfa...


In [73]:
# =============================================================================
# Validate persisted notebook-600 artifact interfaces
# =============================================================================

roundtrip_records = []

for artifact_id, artifact in artifact_payloads.items():
    path = artifact["path"]
    expected = artifact["data"]
    artifact_format = artifact["format"]

    manifest_row = (
        phase6_600_artifact_manifest
        .loc[
            lambda data: data["artifact_id"].eq(artifact_id)
        ]
        .iloc[0]
    )

    assert path.exists()
    assert path.stat().st_size == manifest_row["size_bytes"]
    assert sha256_file(path) == manifest_row["sha256"]

    if artifact_format == "parquet":
        observed = pd.read_parquet(path)

        pd.testing.assert_frame_equal(
            observed,
            expected,
            check_exact=True,
        )

    elif artifact_format == "csv":
        observed = pd.read_csv(path)

        # CSV type inference may differ from the in-memory representation;
        # validate the persisted interface rather than re-testing upstream biology.
        assert list(observed.columns) == list(expected.columns)
        assert len(observed) == len(expected)

    elif artifact_format == "json":
        with path.open(
            "r",
            encoding="utf-8",
        ) as handle:
            observed = json.load(handle)

        assert observed == expected

    else:
        raise ValueError(
            f"Unsupported artifact format: {artifact_format}"
        )

    roundtrip_records.append(
        {
            "artifact_id": artifact_id,
            "format": artifact_format,
            "status": "PASS",
        }
    )

phase6_600_roundtrip_summary = (
    pd.DataFrame(roundtrip_records)
    .sort_values("artifact_id")
    .reset_index(drop=True)
)

assert len(phase6_600_roundtrip_summary) == 9

print(
    "Persisted artifact interfaces validated:",
    len(phase6_600_roundtrip_summary),
)

display(
    phase6_600_roundtrip_summary
)

Persisted artifact interfaces validated: 9


,artifact_id,format,status
0,phase6.600.analysis_metadata,json,PASS
1,phase6.600.cross_resource_compound_catalog,csv,PASS
2,phase6.600.ctrp_analysis_universe,parquet,PASS
3,phase6.600.ctrp_model_crosswalk,csv,PASS
4,phase6.600.drug_eligibility,csv,PASS
5,phase6.600.gdsc_analysis_universe,parquet,PASS
6,phase6.600.gdsc_program_drug_associations,parquet,PASS
7,phase6.600.prism_analysis_universe,parquet,PASS
8,phase6.600.program_score_universe,parquet,PASS


In [74]:
# =============================================================================
# Define notebook-600 artifact registry provenance
# =============================================================================

artifact_registry_path = (
    Paths.root
    / "config"
    / "artifact_registry.json"
)

producer = {
    "type": "notebook",
    "path": (
        "notebooks/phase6_pharmacogenomic_contexts/"
        "600_program_drug_associations.ipynb"
    ),
}


def artifact_input(artifact_id):
    return {
        "type": "artifact",
        "artifact_id": artifact_id,
    }


def raw_input(ref):
    return {
        "type": "raw_registry",
        "ref": ref,
    }


DEPMAP_MODEL = "/depmap/files/Model.csv"
DEPMAP_EXPRESSION = (
    "/depmap/files/"
    "OmicsExpressionProteinCodingGenesTPMLogp1.csv"
)
GDSC_RESPONSE = (
    "/gdsc/files/"
    "GDSC2_fitted_dose_response_27Oct23.xlsx"
)
CTRP_ARCHIVE = (
    "/ctrp/files/"
    "CTRPv2.0_2015_ctd2_ExpandedDataset.zip"
)
PRISM_CELL_INFO = (
    "/prism/files/"
    "secondary-screen-cell-line-info.csv"
)
PRISM_RESPONSE = (
    "/prism/files/"
    "secondary-screen-dose-response-curve-parameters.csv"
)
PRISM_TREATMENT = (
    "/prism/files/"
    "secondary-screen-replicate-collapsed-treatment-info.csv"
)

A = artifact_input
R = raw_input

phase6_registry_inputs = {
    "phase6.600.program_score_universe": [
        A("phase4.401.consensus_cellline_scores"),
        A("phase4.401.consensus_transcriptomic_gene_weights"),
        A("phase3.303.harmonized_expression"),
        R(DEPMAP_MODEL),
        R(DEPMAP_EXPRESSION),
    ],
    "phase6.600.ctrp_model_crosswalk": [
        A("phase6.600.program_score_universe"),
        R(CTRP_ARCHIVE),
        R(DEPMAP_MODEL),
    ],
    "phase6.600.cross_resource_compound_catalog": [
        A("phase6.600.program_score_universe"),
        R(GDSC_RESPONSE),
        R(CTRP_ARCHIVE),
        R(PRISM_RESPONSE),
        R(PRISM_TREATMENT),
    ],
    "phase6.600.drug_eligibility": [
        A("phase6.600.program_score_universe"),
        A("phase6.600.ctrp_model_crosswalk"),
        A("phase6.600.cross_resource_compound_catalog"),
        R(GDSC_RESPONSE),
        R(CTRP_ARCHIVE),
        R(PRISM_RESPONSE),
    ],
    "phase6.600.gdsc_analysis_universe": [
        A("phase6.600.program_score_universe"),
        A("phase6.600.drug_eligibility"),
        A("phase3.302.integrated_modeling_cohort"),
        R(GDSC_RESPONSE),
    ],
    "phase6.600.ctrp_analysis_universe": [
        A("phase6.600.program_score_universe"),
        A("phase6.600.ctrp_model_crosswalk"),
        A("phase6.600.drug_eligibility"),
        R(CTRP_ARCHIVE),
    ],
    "phase6.600.prism_analysis_universe": [
        A("phase6.600.program_score_universe"),
        A("phase6.600.cross_resource_compound_catalog"),
        A("phase6.600.drug_eligibility"),
        R(PRISM_CELL_INFO),
        R(PRISM_RESPONSE),
        R(PRISM_TREATMENT),
    ],
    "phase6.600.gdsc_program_drug_associations": [
        A("phase6.600.gdsc_analysis_universe"),
        A("phase6.600.program_score_universe"),
        R(GDSC_RESPONSE),
    ],
    "phase6.600.analysis_metadata": [
        A("phase6.600.program_score_universe"),
        A("phase6.600.ctrp_model_crosswalk"),
        A("phase6.600.cross_resource_compound_catalog"),
        A("phase6.600.drug_eligibility"),
        A("phase6.600.gdsc_analysis_universe"),
        A("phase6.600.ctrp_analysis_universe"),
        A("phase6.600.prism_analysis_universe"),
        A("phase6.600.gdsc_program_drug_associations"),
    ],
}

assert set(phase6_registry_inputs) == set(
    artifact_payloads
)

In [75]:
# =============================================================================
# Register stable notebook-600 artifacts
# =============================================================================

with artifact_registry_path.open(
    "r",
    encoding="utf-8",
) as handle:
    artifact_registry = json.load(handle)

assert artifact_registry["schema_version"] == 1

manifest_by_id = (
    phase6_600_artifact_manifest
    .set_index("artifact_id")
)

phase6_registry_entries = {}

for artifact_id, artifact in artifact_payloads.items():
    manifest_row = manifest_by_id.loc[
        artifact_id
    ]
    data = artifact["data"]

    phase6_registry_entries[artifact_id] = {
        "path": manifest_row["path"],
        "phase": 6,
        "status": "frozen",
        "artifact_role": (
            "metadata"
            if artifact_id
            == "phase6.600.analysis_metadata"
            else "handoff"
        ),
        "producer": producer,
        "shape": (
            list(data.shape)
            if isinstance(data, pd.DataFrame)
            else None
        ),
        "size_bytes": int(
            manifest_row["size_bytes"]
        ),
        "sha256": manifest_row["sha256"],
        "inputs": phase6_registry_inputs[
            artifact_id
        ],
    }

known_artifact_ids = (
    set(artifact_registry["artifacts"])
    | set(phase6_registry_entries)
)

for artifact_id, entry in (
    phase6_registry_entries.items()
):
    referenced_artifacts = {
        record["artifact_id"]
        for record in entry["inputs"]
        if record["type"] == "artifact"
    }

    assert referenced_artifacts <= known_artifact_ids

added_artifacts = []

for artifact_id, entry in (
    phase6_registry_entries.items()
):
    existing = artifact_registry[
        "artifacts"
    ].get(artifact_id)

    if existing is None:
        artifact_registry[
            "artifacts"
        ][artifact_id] = entry

        added_artifacts.append(
            artifact_id
        )
    else:
        assert existing == entry, (
            f"Existing registry entry differs: "
            f"{artifact_id}"
        )

if added_artifacts:
    with artifact_registry_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        json.dump(
            artifact_registry,
            handle,
            ensure_ascii=False,
            indent=2,
        )
        handle.write("\n")

with artifact_registry_path.open(
    "r",
    encoding="utf-8",
) as handle:
    registered = json.load(handle)

assert all(
    registered["artifacts"][artifact_id]
    == entry
    for artifact_id, entry
    in phase6_registry_entries.items()
)

print(
    "Notebook-600 registry entries:",
    len(phase6_registry_entries),
)
print(
    "New entries added:",
    len(added_artifacts),
)

Notebook-600 registry entries: 9
New entries added: 0


## 8. Notebook 600 closure

This notebook established the first pharmacogenomic layer of Phase 6 and defined the analysis and handoff objects required for downstream notebooks.

In GDSC, 843 associations spanning 281 eligible drugs and the three frozen consensus transcriptomic programs were evaluated using lineage-adjusted models (`LN_IC50 ~ program_score + C(OncotreeLineage)`), with HC3 robust standard errors, two-sided t-based inference, and joint Benjamini–Hochberg correction across all 843 tests. A total of 338 associations met `q < 0.05`, corresponding to 222 unique drugs.

These results must be interpreted as **developmental/internal pharmacogenomic associations**, not as independent validation. GDSC contributed upstream to the construction of the cell-line context used to derive the consensus programs; therefore, the evidence reported here does not constitute independent cross-screen replication.

Leave-one-lineage-out sensitivity analysis showed that 175 of the 843 associations changed sign after omission of at least one supported lineage. Among the 338 FDR-controlled associations, only 4 showed any sign reversal, and each retained the primary direction in 10 of 11 refits. This pattern supports general directional stability within the FDR-controlled set, but it does not establish homogeneity across lineages or independence from cohort composition.

The analysis universes required for external evaluation in CTRP and PRISM were also constructed. Program scores were projected to external models using only the frozen Phase 4 transformation parameters, with no re-standardization in external or combined datasets. CTRP and PRISM remain reserved as external evidence sources; program–drug association results from those resources were not inspected in this notebook.

Cross-resource harmonization produced 228 candidate exact-name links based on unambiguous normalized compound names across at least two resources. These mappings should be treated as candidate computational identity links rather than definitive evidence of complete chemical identity, because differences in salt form, stereochemistry, formulation, or other properties may not be represented by the normalized name.

GDSC `PUTATIVE_TARGET` and `PATHWAY_NAME` annotations were retained exclusively as descriptive pharmacological context. They were not used to alter the multiplicity family, rescue nominal associations, define chemical drug-family identity, or retrospectively prioritize results.

No frozen and methodologically defensible proliferation representation with appropriate coverage was available for inclusion as a covariate in this analysis. Residual proliferation confounding therefore remains an explicit limitation, and no post hoc proxy was introduced.

Nine stable notebook-600 interfaces were persisted and validated, including the program-score universe, model crosswalks, cross-resource compound catalog, drug-eligibility table, GDSC/CTRP/PRISM analysis universes, complete GDSC association results, and methodological metadata. Persisted artifacts passed round-trip validation and were registered in `config/artifact_registry.json`.

### Operational interpretation

Notebook 600 does not select drugs, targets, or therapeutic hypotheses. Its role is to define the pharmacogenomic analysis context reproducibly, characterize developmental/internal GDSC program–drug associations, and establish a frozen analytical boundary for subsequent Phase 6 evaluation. CTRP and PRISM evidence must be analyzed under prospectively frozen rules and must not be used to retroactively modify decisions already evaluated in GDSC.